#ABC FULL

In [ ]:
# =====================================================================
# ELAPS — SANTANDER CANONICAL A/B/C + S0/S1
# UNIFIED MISSING-VALUE NORMALIZATION
#
# Mục tiêu:
#   1. Đọc lại trực tiếp Santander_CreditCard.csv
#   2. Chuẩn hóa missing MỘT LẦN trước mọi aggregation
#   3. A / B / C dùng cùng dữ liệu đã chuẩn hóa
#   4. Cùng khách hàng, nhãn, split
#   5. Preprocessing chỉ fit trên training rows / training fold
#   6. Chạy S0 canonical
#   7. Chạy S1 sensitivity: loại khách không có snapshot sau day 60
#   8. Xuất metrics, ranking, bootstrap, DeLong, S14-S17
#
# Không đọc model/checkpoint A/B/C cũ.
# =====================================================================


# =====================================================================
# 0. CÀI GÓI NẾU THIẾU
# =====================================================================

import importlib.util
import subprocess
import sys

for module, package in [
    ("duckdb", "duckdb"),
    ("xgboost", "xgboost"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package,
            ]
        )


# =====================================================================
# 1. IMPORT
# =====================================================================

from datetime import datetime, timezone
from pathlib import Path
from hashlib import sha256

import gc
import json
import platform
import shutil

import duckdb
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost

from scipy.stats import norm, rankdata

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBClassifier


# =====================================================================
# 2. GOOGLE DRIVE
# =====================================================================

try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass


# =====================================================================
# 3. CẤU HÌNH
# =====================================================================

ROOT = Path("/content/drive/MyDrive")

RAW_FILE = (
    ROOT
    / "Santander_CreditCard.csv"
)

OUTPUT_PARENT = (
    ROOT
    / "Santander_ABC_UnifiedNormalization"
)

WORK_PARENT = Path("/content")


# -----------------------------------------------------
# Model
# -----------------------------------------------------

XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    max_bin=256,
    random_state=42,
    n_jobs=2,
)


# -----------------------------------------------------
# Seeds
# -----------------------------------------------------

SPLIT_SEED = 42
CV_SEED = 42

N_FOLDS = 5

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 20260927

REFIT_BOOTSTRAP_SEED = 20261012


# =====================================================================
# 4. 20 PREDICTORS SANTANDER
# =====================================================================

FEATURES = [
    "IND_EMPLEADO",
    "PAIS_RESIDENCIA",
    "SEXO",
    "AGE",
    "IND_NUEVO",
    "ANTIGUEDAD",
    "INDREL",
    "INDREL_1MES",
    "TIPREL_1MES",
    "INDRESI",
    "INDEXT",
    "CONYUEMP",
    "INDFALL",
    "TIPODOM",
    "COD_PROV",
    "NOMPROV",
    "IND_ACTIVIDAD_CLIENTE",
    "RENTA",
    "SEGMENTO",
    "CANAL_ENTRADA",
]

assert len(FEATURES) == 20
assert len(set(FEATURES)) == 20


# =====================================================================
# 5. QUY TẮC MISSING CHUNG CHO A/B/C
# =====================================================================

# Các token này đều trở thành NULL ngay khi đọc raw.
#
# So sánh không phân biệt hoa/thường vì dùng UPPER(TRIM()).

MISSING_SQL_VALUES = (
    "'', "
    "'NA', "
    "'N/A', "
    "'NAN', "
    "'NULL', "
    "'NONE'"
)


MISSING_PYTHON_VALUES = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NAN",
    "nan",
    "NULL",
    "null",
    "None",
    "none",
]


def clean_sql_expression(expression):
    """
    Một quy tắc duy nhất dùng cho mọi raw feature.

    Ví dụ:
        ""
        " "
        "NA"
        "N/A"
        "nan"
        "NULL"
        "None"

    => SQL NULL

    Các giá trị hợp lệ được TRIM whitespace.
    """

    return (
        "CASE "
        f"WHEN {expression} IS NULL THEN NULL "
        f"WHEN UPPER(TRIM({expression})) "
        f"IN ({MISSING_SQL_VALUES}) THEN NULL "
        f"ELSE TRIM({expression}) "
        "END"
    )


# =====================================================================
# 6. HELPER
# =====================================================================

def sql_string(value):
    return (
        "'"
        + str(value).replace(
            "'",
            "''",
        )
        + "'"
    )


def sha_file(path):

    digest = sha256()

    with Path(path).open(
        "rb"
    ) as stream:

        for piece in iter(
            lambda: stream.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(piece)

    return digest.hexdigest()


# =====================================================================
# 7. XÂY DỰNG S0 A/B/C TỪ RAW
# =====================================================================

def build_first_year_abc(
    out,
    work,
):

    """
    Xây S0 canonical.

    QUAN TRỌNG:
    - Raw snapshot được normalize missing trước mọi aggregation.
    - A/B/C dùng CHUNG first_year_data.
    - Không có missing-rule riêng cho B.
    """

    if not RAW_FILE.is_file():

        raise FileNotFoundError(
            f"Không thấy file:\n{RAW_FILE}"
        )


    # -----------------------------------------------------------------
    # DuckDB
    # -----------------------------------------------------------------

    con = duckdb.connect(
        str(
            work
            / "first_year.duckdb"
        )
    )

    con.execute(
        "SET memory_limit='4GB'"
    )

    con.execute(
        "SET threads=2"
    )

    spill = (
        work
        / "spill"
    )

    spill.mkdir(
        exist_ok=True
    )

    con.execute(
        f"SET temp_directory="
        f"{sql_string(spill)}"
    )


    source = (
        f"read_csv("
        f"{sql_string(RAW_FILE)}, "
        f"header=true, "
        f"all_varchar=true, "
        f"sample_size=10000, "
        f"ignore_errors=true"
        f")"
    )


    try:

        # =============================================================
        # 7.1 KIỂM TRA COLUMN
        # =============================================================

        desc = con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM {source}
            LIMIT 1
            """
        ).fetchdf()


        columns = set(
            desc[
                "column_name"
            ]
            .str.lower()
        )


        required_core = {
            "ncodpers",
            "fecha_dato",
            "fecha_alta",
            "ind_tjcr_fin_ult1",
        }


        missing_core = (
            required_core
            - columns
        )


        if missing_core:

            raise ValueError(
                "CSV thiếu cột bắt buộc: "
                + str(
                    sorted(
                        missing_core
                    )
                )
            )


        missing_features = [
            feature
            for feature
            in FEATURES
            if feature.lower()
            not in columns
        ]


        # Không tự tạo cột toàn NULL.
        if missing_features:

            raise ValueError(
                "CSV thiếu predictor Santander: "
                + ", ".join(
                    missing_features
                )
            )


        # =============================================================
        # 7.2 FEATURE SQL — MỘT RULE CHUNG
        # =============================================================

        feature_sql = ",\n".join(

            f"""
            {clean_sql_expression(
                f's."{feature.lower()}"'
            )}
            AS "{feature}"
            """

            for feature
            in FEATURES
        )


        target_clean_sql = (
            clean_sql_expression(
                's.ind_tjcr_fin_ult1'
            )
        )


        # =============================================================
        # 7.3 RAW DATA
        #
        # Missing feature + target đều normalize ở đây.
        # Sau bước này A/B/C không có rule missing riêng.
        # =============================================================

        con.execute(
            f"""
            CREATE TABLE raw_data AS

            SELECT

                TRY_CAST(
                    NULLIF(
                        TRIM(s.ncodpers),
                        ''
                    )
                    AS BIGINT
                )
                AS ncodpers,


                TRY_CAST(
                    NULLIF(
                        TRIM(s.fecha_dato),
                        ''
                    )
                    AS DATE
                )
                AS fecha_dato,


                TRY_CAST(
                    NULLIF(
                        TRIM(s.fecha_alta),
                        ''
                    )
                    AS DATE
                )
                AS fecha_alta,


                TRY_CAST(
                    {target_clean_sql}
                    AS DOUBLE
                )
                AS card_status,


                {feature_sql}

            FROM {source} AS s

            WHERE

                TRY_CAST(
                    NULLIF(
                        TRIM(s.ncodpers),
                        ''
                    )
                    AS BIGINT
                )
                IS NOT NULL

                AND

                TRY_CAST(
                    NULLIF(
                        TRIM(s.fecha_dato),
                        ''
                    )
                    AS DATE
                )
                IS NOT NULL

                AND

                TRY_CAST(
                    NULLIF(
                        TRIM(s.fecha_alta),
                        ''
                    )
                    AS DATE
                )
                IS NOT NULL
            """
        )


        # =============================================================
        # 7.4 TARGET AUDIT
        # =============================================================

        target_audit = (
            con.execute(
                """
                SELECT

                    COUNT(*) AS n_rows,

                    COUNT(*)
                    FILTER (
                        WHERE card_status IS NULL
                    )
                    AS n_missing_target,

                    COUNT(*)
                    FILTER (
                        WHERE card_status IS NOT NULL
                        AND card_status NOT IN (0,1)
                    )
                    AS n_invalid_target

                FROM raw_data
                """
            )
            .fetchdf()
            .iloc[0]
        )


        print(
            "\n"
            + "=" * 80
        )

        print(
            "TARGET AUDIT"
        )

        print(
            "=" * 80
        )

        print(
            target_audit.to_string()
        )


        if int(
            target_audit[
                "n_missing_target"
            ]
        ) > 0:

            raise ValueError(
                "Có snapshot thiếu trạng thái thẻ. "
                "Không mặc định missing thành 0."
            )


        if int(
            target_audit[
                "n_invalid_target"
            ]
        ) > 0:

            raise ValueError(
                "ind_tjcr_fin_ult1 có "
                "giá trị ngoài {0,1}."
            )


        # =============================================================
        # 7.5 FIRST RELATIONSHIP DATE
        # =============================================================

        start_date_audit = (
            con.execute(
                """
                SELECT COUNT(*)
                FROM (

                    SELECT
                        ncodpers,
                        COUNT(
                            DISTINCT fecha_alta
                        ) AS n_start_dates

                    FROM raw_data

                    GROUP BY
                        ncodpers

                    HAVING
                        COUNT(
                            DISTINCT fecha_alta
                        ) > 1
                )
                """
            )
            .fetchone()[0]
        )


        print(
            "\nCustomers with multiple "
            "fecha_alta:",
            start_date_audit,
        )


        # First relationship start.
        con.execute(
            """
            CREATE TABLE normalized AS

            SELECT

                * EXCLUDE(
                    fecha_alta,
                    card_status
                ),

                MIN(
                    fecha_alta
                )
                OVER (
                    PARTITION BY
                        ncodpers
                )
                AS fecha_alta,

                CASE
                    WHEN card_status > 0
                    THEN 1
                    ELSE 0
                END
                AS has_card

            FROM raw_data
            """
        )


        # =============================================================
        # 7.6 SOURCE STATISTICS
        # =============================================================

        stats = (
            con.execute(
                """
                SELECT

                    COUNT(*) AS valid_source_rows,

                    COUNT(*)
                    FILTER (
                        WHERE
                            fecha_dato
                            < fecha_alta
                    )
                    AS before_relationship_rows,

                    COUNT(*)
                    FILTER (
                        WHERE
                            fecha_dato
                            >= fecha_alta
                               + INTERVAL '1 year'
                    )
                    AS rows_at_or_after_first_year,

                    COUNT(*)
                    FILTER (
                        WHERE
                            fecha_dato
                            >= fecha_alta

                            AND

                            fecha_dato
                            < fecha_alta
                              + INTERVAL '1 year'
                    )
                    AS first_year_rows,

                    MIN(
                        fecha_dato
                    )
                    AS source_first_date,

                    MAX(
                        fecha_dato
                    )
                    AS source_last_date

                FROM normalized
                """
            )
            .fetchdf()
            .iloc[0]
            .to_dict()
        )


        if not stats[
            "valid_source_rows"
        ]:

            raise ValueError(
                "Không có dòng hợp lệ."
            )


        # Exclusive administrative endpoint.
        admin_end = (
            pd.Timestamp(
                stats[
                    "source_last_date"
                ]
            )
            + pd.Timedelta(
                days=1
            )
        )


        # =============================================================
        # 7.7 FIRST-YEAR SNAPSHOT SOURCE CHUNG
        # =============================================================

        con.execute(
            """
            CREATE TABLE first_year_data AS

            SELECT
                *

            FROM normalized

            WHERE
                fecha_dato
                >= fecha_alta

                AND

                fecha_dato
                < fecha_alta
                  + INTERVAL '1 year'
            """
        )


        dup = con.execute(
            """
            SELECT

                ncodpers,
                fecha_dato,
                COUNT(*) AS n

            FROM first_year_data

            GROUP BY
                1,2

            HAVING
                COUNT(*) > 1

            LIMIT 1
            """
        ).fetchone()


        if dup:

            raise ValueError(
                "Có snapshot trùng "
                f"ID/ngày: {dup}"
            )


        # =============================================================
        # 7.8 A FEATURE AGGREGATION
        # =============================================================

        last_features = (
            FEATURES[:-1]
        )


        a_expr = ",\n".join(

            f"""
            ARG_MAX(
                "{feature}",
                fecha_dato
            )
            FILTER (
                WHERE
                    "{feature}"
                    IS NOT NULL
            )
            AS "{feature}"
            """

            for feature
            in last_features
        )


        # Entry channel = first observed valid channel.
        a_expr += """
        ,
        ARG_MIN(
            "CANAL_ENTRADA",
            fecha_dato
        )
        FILTER (
            WHERE
                "CANAL_ENTRADA"
                IS NOT NULL
        )
        AS "CANAL_ENTRADA"
        """


        # =============================================================
        # 7.9 BRANCH A — DAY 60
        # =============================================================

        con.execute(
            f"""
            CREATE TABLE a_window AS

            SELECT

                ncodpers,

                fecha_alta,

                CAST(
                    fecha_alta
                    + INTERVAL '60 days'
                    AS DATE
                )
                AS cutoff_date,

                CAST(
                    fecha_alta
                    + INTERVAL '1 year'
                    AS DATE
                )
                AS year_end_exclusive,

                MAX(
                    fecha_dato
                )
                AS last_feature_snapshot,

                DATE_DIFF(
                    'day',
                    fecha_alta,
                    MAX(fecha_dato)
                )
                AS observed_feature_days,

                COUNT(*)
                AS n_feature_snapshots,

                MAX(
                    has_card
                )
                AS adopted_within_60d,

                {a_expr}

            FROM first_year_data

            WHERE

                fecha_dato
                >= fecha_alta

                AND

                fecha_dato
                <= fecha_alta
                   + INTERVAL '60 days'

            GROUP BY
                ncodpers,
                fecha_alta
            """
        )


        # =============================================================
        # 7.10 POST-60 LABEL
        # =============================================================

        con.execute(
            """
            CREATE TABLE after_60 AS

            SELECT

                ncodpers,

                MAX(
                    has_card
                )::INTEGER
                AS target_60d,

                MIN(
                    fecha_dato
                )
                FILTER (
                    WHERE
                        has_card = 1
                )
                AS first_card_date_after_cutoff,

                MAX(
                    fecha_dato
                )
                AS last_available_snapshot,

                COUNT(*)
                AS n_followup_snapshots

            FROM first_year_data

            WHERE

                fecha_dato
                > fecha_alta
                  + INTERVAL '60 days'

            GROUP BY
                ncodpers
            """
        )


        # =============================================================
        # 7.11 A COHORT
        # =============================================================

        feature_select_a = ", ".join(
            f'a."{feature}"'
            for feature
            in FEATURES
        )


        con.execute(
            f"""
            CREATE TABLE a_cohort AS

            SELECT

                a.ncodpers
                AS "NCODPERS",

                a.fecha_alta
                AS "FECHA_ALTA",

                a.cutoff_date
                AS "CUTOFF_DATE",

                a.year_end_exclusive
                AS "YEAR_END_EXCLUSIVE",

                a.last_feature_snapshot
                AS "LAST_FEATURE_SNAPSHOT",

                a.observed_feature_days
                AS "OBSERVED_FEATURE_DAYS",

                a.n_feature_snapshots
                AS "N_FEATURE_SNAPSHOTS",

                t.first_card_date_after_cutoff
                AS "FIRST_CARD_DATE_AFTER_CUTOFF",

                t.last_available_snapshot
                AS "LAST_AVAILABLE_SNAPSHOT",

                COALESCE(
                    t.n_followup_snapshots,
                    0
                )
                AS "N_FOLLOWUP_SNAPSHOTS",

                {feature_select_a},

                COALESCE(
                    t.target_60d,
                    0
                )::INTEGER
                AS "TARGET_60D"

            FROM
                a_window a

            LEFT JOIN
                after_60 t
            USING (
                ncodpers
            )

            WHERE
                a.adopted_within_60d = 0
            """
        )


        audit = (
            con.execute(
                """
                SELECT *
                FROM a_cohort
                ORDER BY "NCODPERS"
                """
            )
            .fetchdf()
        )


        if (
            audit.empty
            or audit[
                "TARGET_60D"
            ].nunique()
            != 2
        ):

            raise ValueError(
                "Cohort rỗng hoặc "
                "target chỉ có một lớp."
            )


        if audit[
            "NCODPERS"
        ].duplicated().any():

            raise AssertionError(
                "Một khách hàng xuất hiện "
                "nhiều hơn một lần."
            )


        # =============================================================
        # 7.12 A AUDIT
        # =============================================================

        if not (
            (
                audit[
                    "CUTOFF_DATE"
                ]
                - audit[
                    "FECHA_ALTA"
                ]
            ).dt.days
            == 60
        ).all():

            raise AssertionError(
                "A cutoff không đúng day 60."
            )


        if not (
            audit[
                "LAST_FEATURE_SNAPSHOT"
            ]
            <= audit[
                "CUTOFF_DATE"
            ]
        ).all():

            raise AssertionError(
                "A có feature sau day 60."
            )


        positives = (
            audit[
                "TARGET_60D"
            ].eq(1)
        )


        if not (
            audit.loc[
                positives,
                "FIRST_CARD_DATE_AFTER_CUTOFF",
            ]
            >
            audit.loc[
                positives,
                "CUTOFF_DATE",
            ]
        ).all():

            raise AssertionError(
                "Positive event nằm "
                "trong A feature window."
            )


        if not (
            audit.loc[
                positives,
                "FIRST_CARD_DATE_AFTER_CUTOFF",
            ]
            <
            audit.loc[
                positives,
                "YEAR_END_EXCLUSIVE",
            ]
        ).all():

            raise AssertionError(
                "Positive event nằm ngoài "
                "first relationship year."
            )


        # =============================================================
        # 7.13 BRANCH B
        #
        # RẤT QUAN TRỌNG:
        # Không còn b_value().
        # B dùng trực tiếp first_year_data đã normalize.
        # =============================================================

        b_cols = ",\n".join(

            f"""
            ARG_MAX(
                s."{feature}",
                s.fecha_dato
            )
            FILTER (
                WHERE
                    s."{feature}"
                    IS NOT NULL
            )
            AS "{feature}"
            """

            for feature
            in last_features
        )


        b_cols += """
        ,
        ARG_MIN(
            s."CANAL_ENTRADA",
            s.fecha_dato
        )
        FILTER (
            WHERE
                s."CANAL_ENTRADA"
                IS NOT NULL
        )
        AS "CANAL_ENTRADA"
        """


        con.execute(
            f"""
            CREATE TABLE b_cohort AS

            WITH cutoffs AS (

                SELECT

                    a."NCODPERS"
                    AS ncodpers,

                    CASE

                        WHEN
                            a."TARGET_60D" = 1

                        THEN
                            a."FIRST_CARD_DATE_AFTER_CUTOFF"

                        ELSE
                            LEAST(
                                a."YEAR_END_EXCLUSIVE",
                                DATE {
                                    sql_string(
                                        admin_end.date()
                                    )
                                }
                            )

                    END
                    AS b_cutoff,

                    a."FIRST_CARD_DATE_AFTER_CUTOFF"
                    AS event_date

                FROM
                    a_cohort a
            )


            SELECT

                s.ncodpers
                AS "NCODPERS",

                c.b_cutoff
                AS "B_CUTOFF",

                c.event_date
                AS "EVENT_DATE",

                MAX(
                    s.fecha_dato
                )
                AS "B_LAST_FEATURE_DATE",

                COUNT(*)
                AS "B_N_FEATURE_SNAPSHOTS",

                {b_cols}

            FROM
                first_year_data s

            JOIN
                cutoffs c
            USING (
                ncodpers
            )

            WHERE

                s.fecha_dato
                < c.b_cutoff

            GROUP BY

                s.ncodpers,
                c.b_cutoff,
                c.event_date
            """
        )


        b = (
            con.execute(
                """
                SELECT *
                FROM b_cohort
                ORDER BY "NCODPERS"
                """
            )
            .fetchdf()
        )


        # =============================================================
        # 7.14 B AUDIT
        # =============================================================

        if not np.array_equal(
            b[
                "NCODPERS"
            ].to_numpy(),
            audit[
                "NCODPERS"
            ].to_numpy(),
        ):

            raise AssertionError(
                "B/C làm thay đổi "
                "cohort hoặc thứ tự."
            )


        if not (
            b[
                "B_LAST_FEATURE_DATE"
            ]
            <
            b[
                "B_CUTOFF"
            ]
        ).all():

            raise AssertionError(
                "B sử dụng snapshot "
                "tại/sau B cutoff."
            )


        if not (
            b[
                "B_N_FEATURE_SNAPSHOTS"
            ].to_numpy()
            >=
            audit[
                "N_FEATURE_SNAPSHOTS"
            ].to_numpy()
        ).all():

            raise AssertionError(
                "B có ít snapshot hơn A."
            )


        # =============================================================
        # 7.15 C TENURE
        # =============================================================

        tenure = (
            b[
                "B_CUTOFF"
            ]
            -
            audit[
                "FECHA_ALTA"
            ]
        ).dt.days


        if (
            tenure.isna().any()
            or (tenure < 0).any()
            or (tenure > 366).any()
        ):

            raise AssertionError(
                "TENURE_AT_CUTOFF "
                "không hợp lệ."
            )


        # =============================================================
        # 7.16 COMMON SPLIT
        # =============================================================

        audit[
            "ROW_ID"
        ] = (
            np.arange(
                1,
                len(audit) + 1,
            )
            .astype(str)
        )


        indices = np.arange(
            len(audit)
        )


        tr, te = train_test_split(

            indices,

            test_size=0.20,

            stratify=audit[
                "TARGET_60D"
            ],

            random_state=SPLIT_SEED,
        )


        audit[
            "SPLIT"
        ] = "train"


        audit.loc[
            te,
            "SPLIT",
        ] = "test"


        # =============================================================
        # 7.17 SAVE SOURCE AUDIT
        # =============================================================

        audit.to_csv(
            out
            / "A_first_year_audit.csv",
            index=False,
        )


        audit[
            FEATURES
            + [
                "TARGET_60D"
            ]
        ].to_csv(
            out
            / "A_first_year_model.csv",
            index=False,
        )


        stats.update(
            dict(
                first_year_cohort=len(
                    audit
                ),
                positives=int(
                    audit[
                        "TARGET_60D"
                    ].sum()
                ),
                n_without_post60_snapshot=int(
                    audit[
                        "N_FOLLOWUP_SNAPSHOTS"
                    ]
                    .eq(0)
                    .sum()
                ),
                max_event_date=audit[
                    "FIRST_CARD_DATE_AFTER_CUTOFF"
                ].max(),
                customers_multiple_start_dates=int(
                    start_date_audit
                ),
            )
        )


        pd.DataFrame(
            [
                stats
            ]
        ).to_csv(
            out
            / "first_year_filter_summary.csv",
            index=False,
        )


        print(
            "\n"
            + "=" * 80
        )

        print(
            "S0 FIRST-YEAR COHORT"
        )

        print(
            "=" * 80
        )

        print(
            f"N           : "
            f"{len(audit):,}"
        )

        print(
            f"Positive    : "
            f"{int(audit.TARGET_60D.sum()):,}"
        )

        print(
            f"Train       : "
            f"{len(tr):,}"
        )

        print(
            f"Holdout     : "
            f"{len(te):,}"
        )

        print(
            "No post60 snapshot:",
            int(
                audit[
                    "N_FOLLOWUP_SNAPSHOTS"
                ]
                .eq(0)
                .sum()
            ),
        )


        return (
            audit,
            b,
            tr,
            te,
            admin_end,
            stats,
            target_audit.to_dict(),
        )


    finally:

        con.close()


# =====================================================================
# 8. PYTHON-SIDE NORMALIZATION
#
# Đây là lớp bảo vệ thứ hai.
# SQL đã normalize trước aggregation.
# Python normalization dùng cùng ý nghĩa.
# =====================================================================

def normalize_missing_frame(
    frame,
):

    x = frame.copy()

    x = x.replace(
        MISSING_PYTHON_VALUES,
        np.nan,
    )

    return x


# =====================================================================
# 9. INFER SCHEMA TỪ A-TRAIN CHỈ MỘT LẦN
#
# Không dùng holdout.
# Không infer riêng A/B/C.
# =====================================================================

def infer_shared_schema(
    x_train,
):

    x = normalize_missing_frame(
        x_train
    )


    numeric_cols = []
    categorical_cols = []


    for column in x.columns:

        converted = pd.to_numeric(
            x[
                column
            ],
            errors="coerce",
        )


        denominator = int(
            x[
                column
            ]
            .notna()
            .sum()
        )


        numeric_count = int(
            converted
            .notna()
            .sum()
        )


        ratio = (
            numeric_count
            / denominator
            if denominator
            else 0.0
        )


        if ratio >= 0.80:

            numeric_cols.append(
                column
            )

        else:

            categorical_cols.append(
                column
            )


    return (
        numeric_cols,
        categorical_cols,
    )


# =====================================================================
# 10. PREPROCESS — FIT TRAIN ONLY
# =====================================================================

def preprocess_with_schema(
    x_train,
    x_valid,
    numeric_cols,
    categorical_cols,
):

    train = normalize_missing_frame(
        x_train
    )

    valid = normalize_missing_frame(
        x_valid
    )


    # -------------------------------------------------------------
    # Numeric
    # -------------------------------------------------------------

    for column in numeric_cols:

        train[
            column
        ] = pd.to_numeric(
            train[
                column
            ],
            errors="coerce",
        )


        valid[
            column
        ] = pd.to_numeric(
            valid[
                column
            ],
            errors="coerce",
        )


    # -------------------------------------------------------------
    # Categorical
    # -------------------------------------------------------------

    if categorical_cols:

        for column in categorical_cols:

            train[
                column
            ] = (
                train[
                    column
                ]
                .astype(
                    "string"
                )
                .fillna(
                    "UNKNOWN"
                )
            )


            valid[
                column
            ] = (
                valid[
                    column
                ]
                .astype(
                    "string"
                )
                .fillna(
                    "UNKNOWN"
                )
            )


        encoder = OrdinalEncoder(

            handle_unknown=
                "use_encoded_value",

            unknown_value=-1,

            encoded_missing_value=-1,

            dtype=np.float32,
        )


        train[
            categorical_cols
        ] = encoder.fit_transform(
            train[
                categorical_cols
            ]
        )


        valid[
            categorical_cols
        ] = encoder.transform(
            valid[
                categorical_cols
            ]
        )


    # -------------------------------------------------------------
    # Ensure numeric matrix
    # -------------------------------------------------------------

    for column in train.columns:

        train[
            column
        ] = pd.to_numeric(
            train[
                column
            ],
            errors="coerce",
        )

        valid[
            column
        ] = pd.to_numeric(
            valid[
                column
            ],
            errors="coerce",
        )


    # -------------------------------------------------------------
    # Imputation fitted TRAIN only
    # -------------------------------------------------------------

    try:

        imputer = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )

    except TypeError:

        imputer = SimpleImputer(
            strategy="median"
        )


    train_array = (
        imputer.fit_transform(
            train
        )
        .astype(
            np.float32
        )
    )


    valid_array = (
        imputer.transform(
            valid
        )
        .astype(
            np.float32
        )
    )


    return (
        train_array,
        valid_array,
    )


# =====================================================================
# 11. METRICS
# =====================================================================

def metrics(
    y,
    scores,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    scores = np.asarray(
        scores,
        dtype=float,
    )


    predicted = (
        scores
        >= 0.5
    ).astype(int)


    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            predicted,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )


    fpr, tpr, _ = (
        roc_curve(
            y,
            scores,
        )
    )


    brier = (
        brier_score_loss(
            y,
            scores,
        )
    )


    prevalence = (
        y.mean()
    )


    denominator = (
        prevalence
        * (
            1
            - prevalence
        )
    )


    bss = (
        1
        - brier
        / denominator

        if denominator > 0

        else np.nan
    )


    return dict(

        AUC=roc_auc_score(
            y,
            scores,
        ),

        KS=float(
            np.max(
                tpr - fpr
            )
        ),

        Recall=recall_score(
            y,
            predicted,
            zero_division=0,
        ),

        Precision=precision_score(
            y,
            predicted,
            zero_division=0,
        ),

        F1=f1_score(
            y,
            predicted,
            zero_division=0,
        ),

        Accuracy=accuracy_score(
            y,
            predicted,
        ),

        Brier=brier,

        BSS=bss,

        Specificity=(
            tn
            / (
                tn + fp
            )
            if (
                tn + fp
            )
            else np.nan
        ),

        NPV=(
            tn
            / (
                tn + fn
            )
            if (
                tn + fn
            )
            else np.nan
        ),

        MCC=matthews_corrcoef(
            y,
            predicted,
        ),

        TP=int(tp),
        FP=int(fp),
        TN=int(tn),
        FN=int(fn),
    )


# =====================================================================
# 12. RANKING
# =====================================================================

def ranking(
    y,
    scores,
    branch,
):

    frame = pd.DataFrame(
        {
            "y":
                np.asarray(y),

            "score":
                np.asarray(
                    scores
                ),
        }
    )


    frame = (
        frame
        .sort_values(
            "score",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    n_positive = int(
        frame[
            "y"
        ].sum()
    )


    prevalence = float(
        frame[
            "y"
        ].mean()
    )


    rows = []


    for pct in (
        10,
        20,
        30,
    ):

        n_contact = max(
            1,
            int(
                np.ceil(
                    len(frame)
                    * pct
                    / 100
                )
            ),
        )


        hits = int(
            frame[
                "y"
            ]
            .iloc[
                :n_contact
            ]
            .sum()
        )


        precision = (
            hits
            / n_contact
        )


        rows.append(
            dict(
                branch=branch,
                top_pct=pct,
                n_contact=n_contact,
                hits=hits,
                precision=precision,
                lift=(
                    precision
                    / prevalence
                ),
                capture=(
                    hits
                    / n_positive
                ),
            )
        )


    return rows


# =====================================================================
# 13. FIT BRANCH
# =====================================================================

def fit_branch(
    x,
    y,
    train_idx,
    test_idx,
    branch,
    base_numeric_cols,
    base_categorical_cols,
    run_cv=True,
):

    x_train = (
        x.iloc[
            train_idx
        ]
        .reset_index(
            drop=True
        )
    )


    x_test = (
        x.iloc[
            test_idx
        ]
        .reset_index(
            drop=True
        )
    )


    y_train = (
        y[
            train_idx
        ]
    )


    y_test = (
        y[
            test_idx
        ]
    )


    # -------------------------------------------------------------
    # C contains one additional numeric variable.
    # -------------------------------------------------------------

    numeric_cols = list(
        base_numeric_cols
    )

    categorical_cols = list(
        base_categorical_cols
    )


    if (
        "TENURE_AT_CUTOFF"
        in x.columns
        and
        "TENURE_AT_CUTOFF"
        not in numeric_cols
    ):

        numeric_cols.append(
            "TENURE_AT_CUTOFF"
        )


    cv_rows = []


    # =============================================================
    # Cross-validation — S0 only
    # =============================================================

    if run_cv:

        cv = StratifiedKFold(

            n_splits=N_FOLDS,

            shuffle=True,

            random_state=CV_SEED,
        )


        for fold, (
            fold_train,
            fold_valid,
        ) in enumerate(

            cv.split(
                x_train,
                y_train,
            ),

            start=1,
        ):

            xt, xv = (
                preprocess_with_schema(

                    x_train.iloc[
                        fold_train
                    ],

                    x_train.iloc[
                        fold_valid
                    ],

                    numeric_cols,

                    categorical_cols,
                )
            )


            model = XGBClassifier(
                **XGB_PARAMS
            )


            model.fit(
                xt,
                y_train[
                    fold_train
                ],
            )


            probability = (
                model.predict_proba(
                    xv
                )[:, 1]
            )


            fold_result = dict(

                branch=branch,

                fold=fold,

                **metrics(
                    y_train[
                        fold_valid
                    ],
                    probability,
                ),
            )


            cv_rows.append(
                fold_result
            )


            print(
                f"{branch} CV "
                f"{fold}/{N_FOLDS}: "
                f"AUC="
                f"{fold_result['AUC']:.6f}"
            )


            del (
                model,
                xt,
                xv,
            )

            gc.collect()


    # =============================================================
    # Final model
    # =============================================================

    xt, xv = (
        preprocess_with_schema(

            x_train,

            x_test,

            numeric_cols,

            categorical_cols,
        )
    )


    model = XGBClassifier(
        **XGB_PARAMS
    )


    model.fit(
        xt,
        y_train,
    )


    scores = (

        model.predict_proba(
            xv
        )[:, 1]

        .astype(float)
    )


    result = dict(

        branch=branch,

        n_train=len(
            train_idx
        ),

        n_test=len(
            test_idx
        ),

        positives_train=int(
            y_train.sum()
        ),

        positives_test=int(
            y_test.sum()
        ),

        **metrics(
            y_test,
            scores,
        ),
    )


    print(

        f"{branch} TEST: "

        f"AUC="
        f"{result['AUC']:.6f}"

        f" | KS="
        f"{result['KS']:.6f}"

        f" | Brier="
        f"{result['Brier']:.6f}"
    )


    rank_rows = ranking(
        y_test,
        scores,
        branch,
    )


    return (
        result,
        cv_rows,
        rank_rows,
        scores,
    )


# =====================================================================
# 14. DELONG
# =====================================================================

def delong_pair(
    y,
    p,
    q,
):

    y = (
        np.asarray(y)
        .astype(bool)
    )


    m = int(
        y.sum()
    )

    n = int(
        (~y).sum()
    )


    s = np.vstack(
        [
            np.asarray(p),
            np.asarray(q),
        ]
    )


    positive = (
        s[
            :,
            y,
        ]
    )

    negative = (
        s[
            :,
            ~y,
        ]
    )


    tx = np.vstack(
        [
            rankdata(
                row,
                method="average",
            )
            for row
            in positive
        ]
    )


    ty = np.vstack(
        [
            rankdata(
                row,
                method="average",
            )
            for row
            in negative
        ]
    )


    tz = np.vstack(
        [
            rankdata(
                row,
                method="average",
            )
            for row
            in np.concatenate(
                [
                    positive,
                    negative,
                ],
                axis=1,
            )
        ]
    )


    auc = (
        (
            tz[
                :,
                :m,
            ].sum(
                axis=1
            )
            / m

            - (
                m + 1
            )
            / 2
        )
        / n
    )


    v01 = (
        tz[
            :,
            :m,
        ]
        - tx
    ) / n


    v10 = (
        1
        -
        (
            tz[
                :,
                m:,
            ]
            - ty
        )
        / m
    )


    covariance = (
        np.cov(
            v01
        )
        / m

        +

        np.cov(
            v10
        )
        / n
    )


    variance = max(

        0.0,

        float(
            covariance[
                0,0
            ]
            +
            covariance[
                1,1
            ]
            -
            2
            * covariance[
                0,1
            ]
        ),
    )


    difference = float(
        auc[0]
        - auc[1]
    )


    if variance == 0:

        return (
            0.0
            if difference
            else 1.0
        )


    return float(

        2
        * norm.sf(

            abs(
                difference
            )
            / np.sqrt(
                variance
            )
        )
    )


# =====================================================================
# 15. PAIRED AUC CONTRAST
# =====================================================================

def contrasts(
    y,
    predictions,
    seed=BOOTSTRAP_SEED,
):

    rng = np.random.default_rng(
        seed
    )


    pairs = [
        (
            "B",
            "A",
        ),
        (
            "C",
            "B",
        ),
        (
            "C",
            "A",
        ),
    ]


    draws = {
        f"{a}-{b}": []
        for a, b
        in pairs
    }


    y = np.asarray(
        y
    )


    n = len(y)


    for i in range(
        N_BOOTSTRAP
    ):

        idx = rng.integers(
            0,
            n,
            size=n,
        )


        sampled_y = (
            y[
                idx
            ]
        )


        if (
            sampled_y.min()
            ==
            sampled_y.max()
        ):
            continue


        aucs = {

            branch:
                roc_auc_score(
                    sampled_y,
                    prediction[
                        idx
                    ],
                )

            for branch,
            prediction
            in predictions.items()
        }


        for a, b in pairs:

            draws[
                f"{a}-{b}"
            ].append(

                aucs[a]
                -
                aucs[b]
            )


    rows = []


    for a, b in pairs:

        name = (
            f"{a}-{b}"
        )


        values = np.asarray(
            draws[
                name
            ]
        )


        rows.append(

            dict(

                contrast=name,

                auc_diff=(
                    roc_auc_score(
                        y,
                        predictions[a],
                    )
                    -
                    roc_auc_score(
                        y,
                        predictions[b],
                    )
                ),

                ci_lower=float(
                    np.quantile(
                        values,
                        0.025,
                    )
                ),

                ci_upper=float(
                    np.quantile(
                        values,
                        0.975,
                    )
                ),

                delong_p=delong_pair(
                    y,
                    predictions[a],
                    predictions[b],
                ),

                bootstrap_valid=len(
                    values
                ),
            )
        )


    return (
        rows,
        pd.DataFrame(
            draws
        ),
    )


# =====================================================================
# 16. AUC CI
# =====================================================================

def bootstrap_auc_ci(
    y,
    scores,
    seed,
):

    rng = np.random.default_rng(
        seed
    )


    y = np.asarray(
        y
    )

    scores = np.asarray(
        scores
    )


    values = []


    for _ in range(
        N_BOOTSTRAP
    ):

        idx = rng.integers(
            0,
            len(y),
            len(y),
        )


        ys = y[
            idx
        ]


        if (
            ys.min()
            ==
            ys.max()
        ):
            continue


        values.append(

            roc_auc_score(
                ys,
                scores[
                    idx
                ],
            )
        )


    return (
        float(
            np.quantile(
                values,
                0.025,
            )
        ),
        float(
            np.quantile(
                values,
                0.975,
            )
        ),
    )


# =====================================================================
# 17. S1 REFIT EFFECTS
# =====================================================================

def bootstrap_refit_effects(
    y,
    old_predictions,
    new_predictions,
):

    rng = np.random.default_rng(
        REFIT_BOOTSTRAP_SEED
    )


    y = np.asarray(
        y
    )


    metric_names = [
        "AUC A",
        "AUC B",
        "AUC C",
        "B-A",
        "C-B",
        "C-A",
    ]


    draws = {
        name: []
        for name
        in metric_names
    }


    def auc_dict(
        labels,
        predictions,
    ):

        return {

            branch:
                roc_auc_score(
                    labels,
                    score,
                )

            for branch,
            score
            in predictions.items()
        }


    observed_old = (
        auc_dict(
            y,
            old_predictions,
        )
    )


    observed_new = (
        auc_dict(
            y,
            new_predictions,
        )
    )


    observed = {

        "AUC A":
            observed_new["A"]
            -
            observed_old["A"],

        "AUC B":
            observed_new["B"]
            -
            observed_old["B"],

        "AUC C":
            observed_new["C"]
            -
            observed_old["C"],

        "B-A":
            (
                observed_new["B"]
                -
                observed_new["A"]
            )
            -
            (
                observed_old["B"]
                -
                observed_old["A"]
            ),

        "C-B":
            (
                observed_new["C"]
                -
                observed_new["B"]
            )
            -
            (
                observed_old["C"]
                -
                observed_old["B"]
            ),

        "C-A":
            (
                observed_new["C"]
                -
                observed_new["A"]
            )
            -
            (
                observed_old["C"]
                -
                observed_old["A"]
            ),
    }


    for _ in range(
        N_BOOTSTRAP
    ):

        idx = rng.integers(
            0,
            len(y),
            len(y),
        )


        ys = y[
            idx
        ]


        if (
            ys.min()
            ==
            ys.max()
        ):
            continue


        old_auc = {

            branch:
                roc_auc_score(
                    ys,
                    score[
                        idx
                    ],
                )

            for branch,
            score
            in old_predictions.items()
        }


        new_auc = {

            branch:
                roc_auc_score(
                    ys,
                    score[
                        idx
                    ],
                )

            for branch,
            score
            in new_predictions.items()
        }


        draws[
            "AUC A"
        ].append(
            new_auc["A"]
            -
            old_auc["A"]
        )


        draws[
            "AUC B"
        ].append(
            new_auc["B"]
            -
            old_auc["B"]
        )


        draws[
            "AUC C"
        ].append(
            new_auc["C"]
            -
            old_auc["C"]
        )


        draws[
            "B-A"
        ].append(
            (
                new_auc["B"]
                -
                new_auc["A"]
            )
            -
            (
                old_auc["B"]
                -
                old_auc["A"]
            )
        )


        draws[
            "C-B"
        ].append(
            (
                new_auc["C"]
                -
                new_auc["B"]
            )
            -
            (
                old_auc["C"]
                -
                old_auc["B"]
            )
        )


        draws[
            "C-A"
        ].append(
            (
                new_auc["C"]
                -
                new_auc["A"]
            )
            -
            (
                old_auc["C"]
                -
                old_auc["A"]
            )
        )


    rows = []


    for name in metric_names:

        values = np.asarray(
            draws[
                name
            ]
        )


        rows.append(

            dict(

                Metric=name,

                Change=observed[
                    name
                ],

                CI_lower=float(
                    np.quantile(
                        values,
                        0.025,
                    )
                ),

                CI_upper=float(
                    np.quantile(
                        values,
                        0.975,
                    )
                ),
            )
        )


    return pd.DataFrame(
        rows
    )


# =====================================================================
# 18. MAIN
# =====================================================================

def main():

    stamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%dT%H%M%S%fZ"
        )
    )


    out = (
        OUTPUT_PARENT
        /
        f"run_{stamp}"
    )


    out.mkdir(
        parents=True,
        exist_ok=False,
    )


    work = (
        WORK_PARENT
        /
        f"santander_unified_work_{stamp}"
    )


    work.mkdir(
        parents=True,
        exist_ok=False,
    )


    print(
        "=" * 80
    )

    print(
        "SANTANDER A/B/C — "
        "UNIFIED NORMALIZATION"
    )

    print(
        "=" * 80
    )

    print(
        "Raw:",
        RAW_FILE,
    )

    print(
        "Output:",
        out,
    )


    try:

        # =========================================================
        # BUILD S0
        # =========================================================

        (
            audit,
            b,
            tr,
            te,
            admin_end,
            source_stats,
            target_audit,
        ) = build_first_year_abc(
            out,
            work,
        )


        y = (
            audit[
                "TARGET_60D"
            ]
            .to_numpy(
                dtype=np.int8
            )
        )


        # =========================================================
        # A / B / C FEATURE MATRICES
        # =========================================================

        tenure = (
            b[
                "B_CUTOFF"
            ]
            -
            audit[
                "FECHA_ALTA"
            ]
        ).dt.days


        branch_x = {

            "A":
                audit[
                    FEATURES
                ]
                .copy(),

            "B":
                b[
                    FEATURES
                ]
                .copy(),

            "C":
                b[
                    FEATURES
                ]
                .copy(),
        }


        branch_x[
            "C"
        ][
            "TENURE_AT_CUTOFF"
        ] = (
            tenure
            .astype(
                "int64"
            )
            .to_numpy()
        )


        if not (
            branch_x[
                "C"
            ][
                FEATURES
            ]
            .equals(
                branch_x[
                    "B"
                ][
                    FEATURES
                ]
            )
        ):

            raise AssertionError(
                "C đã thay đổi "
                "các predictors của B."
            )


        # =========================================================
        # SHARED SCHEMA
        #
        # Infer từ A training S0.
        # Reuse cho A/B/C và S1.
        # =========================================================

        (
            base_numeric_cols,
            base_categorical_cols,
        ) = infer_shared_schema(

            branch_x[
                "A"
            ].iloc[
                tr
            ]
        )


        schema = {

            "numeric":
                base_numeric_cols,

            "categorical":
                base_categorical_cols,

            "source":
                "S0 Branch A training rows only",

            "numeric_rule":
                ">=80% parseable numeric among nonmissing training values",

            "missing_rule":
                [
                    "",
                    "whitespace",
                    "NA",
                    "N/A",
                    "NAN",
                    "NULL",
                    "NONE",
                ],
        }


        (
            out
            / "shared_preprocessing_schema.json"
        ).write_text(

            json.dumps(
                schema,
                indent=2,
                ensure_ascii=False,
            ),

            encoding="utf-8",
        )


        print(
            "\n"
            + "=" * 80
        )

        print(
            "SHARED FEATURE SCHEMA"
        )

        print(
            "=" * 80
        )

        print(
            "Numeric:",
            base_numeric_cols,
        )

        print(
            "Categorical:",
            base_categorical_cols,
        )


        # =========================================================
        # SAVE SHARED SPLIT
        # =========================================================

        keys = audit[
            [
                "NCODPERS",
                "ROW_ID",
                "TARGET_60D",
                "SPLIT",
            ]
        ].copy()


        keys.to_csv(
            out
            / "shared_A_split.csv",
            index=False,
        )


        cutoff_audit = (
            keys.copy()
        )


        cutoff_audit[
            "A_CUTOFF"
        ] = audit[
            "CUTOFF_DATE"
        ]


        cutoff_audit[
            "YEAR_END_EXCLUSIVE"
        ] = audit[
            "YEAR_END_EXCLUSIVE"
        ]


        cutoff_audit[
            "B_CUTOFF"
        ] = b[
            "B_CUTOFF"
        ].to_numpy()


        cutoff_audit[
            "EVENT_DATE"
        ] = b[
            "EVENT_DATE"
        ].to_numpy()


        cutoff_audit[
            "B_LAST_FEATURE_DATE"
        ] = b[
            "B_LAST_FEATURE_DATE"
        ].to_numpy()


        cutoff_audit[
            "B_N_FEATURE_SNAPSHOTS"
        ] = b[
            "B_N_FEATURE_SNAPSHOTS"
        ].to_numpy()


        cutoff_audit[
            "N_FOLLOWUP_SNAPSHOTS"
        ] = audit[
            "N_FOLLOWUP_SNAPSHOTS"
        ].to_numpy()


        cutoff_audit[
            "TENURE_AT_CUTOFF"
        ] = (
            tenure
            .to_numpy()
        )


        cutoff_audit.to_csv(
            out
            / "paired_cutoff_audit.csv",
            index=False,
        )


        # =========================================================
        # SAVE MATRICES
        # =========================================================

        for name in (
            "A",
            "B",
            "C",
        ):

            matrix = pd.concat(

                [
                    keys.reset_index(
                        drop=True
                    ),

                    branch_x[
                        name
                    ].reset_index(
                        drop=True
                    ),
                ],

                axis=1,
            )


            matrix.to_csv(

                out
                / f"branch_{name}_matrix.csv",

                index=False,
            )


        # =========================================================
        # S0 FIT A/B/C
        # =========================================================

        print(
            "\n"
            + "=" * 80
        )

        print(
            "S0 — CANONICAL A/B/C"
        )

        print(
            "=" * 80
        )


        s0_results = []
        s0_cv = []
        s0_rankings = []
        s0_predictions = {}


        for name in (
            "A",
            "B",
            "C",
        ):

            (
                result,
                cv_rows,
                rank_rows,
                scores,
            ) = fit_branch(

                branch_x[
                    name
                ],

                y,

                tr,

                te,

                name,

                base_numeric_cols,

                base_categorical_cols,

                run_cv=True,
            )


            s0_results.append(
                result
            )


            s0_cv.extend(
                cv_rows
            )


            s0_rankings.extend(
                rank_rows
            )


            s0_predictions[
                name
            ] = scores


        # =========================================================
        # S0 SAVE METRICS
        # =========================================================

        pd.DataFrame(
            s0_results
        ).to_csv(

            out
            / "S0_holdout_metrics.csv",

            index=False,
        )


        cvdf = pd.DataFrame(
            s0_cv
        )


        cvdf.to_csv(
            out
            / "S0_cv_folds.csv",
            index=False,
        )


        summary_columns = [
            "AUC",
            "KS",
            "Recall",
            "Precision",
            "F1",
            "Accuracy",
            "Brier",
            "BSS",
        ]


        cv_summary = (
            cvdf
            .groupby(
                "branch"
            )[
                summary_columns
            ]
            .agg(
                [
                    "mean",
                    "std",
                ]
            )
        )


        cv_summary.columns = [

            f"{a}_{b}"

            for a, b
            in cv_summary.columns
        ]


        cv_summary = (
            cv_summary
            .reset_index()
        )


        cv_summary.to_csv(
            out
            / "S0_cv_summary.csv",
            index=False,
        )


        pd.DataFrame(
            s0_rankings
        ).to_csv(

            out
            / "S0_ranking_top10_20_30.csv",

            index=False,
        )


        # =========================================================
        # S0 PREDICTIONS
        # =========================================================

        s0_prediction_frame = (

            audit.iloc[
                te
            ][
                [
                    "NCODPERS",
                    "ROW_ID",
                    "TARGET_60D",
                ]
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        for name in (
            "A",
            "B",
            "C",
        ):

            s0_prediction_frame[
                f"SCORE_{name}"
            ] = s0_predictions[
                name
            ]


        s0_prediction_frame.to_csv(

            out
            / "S0_paired_holdout_predictions.csv",

            index=False,
        )


        # =========================================================
        # S0 PAIRED CONTRASTS
        # =========================================================

        y_s0_test = (
            y[
                te
            ]
        )


        (
            s0_contrast_rows,
            s0_bootstrap,
        ) = contrasts(

            y_s0_test,

            s0_predictions,

            seed=
                BOOTSTRAP_SEED,
        )


        pd.DataFrame(
            s0_contrast_rows
        ).to_csv(

            out
            / "S0_paired_auc_contrasts.csv",

            index=False,
        )


        s0_bootstrap.to_csv(

            out
            / "S0_paired_bootstrap_draws.csv",

            index=False,
        )


        # =========================================================
        # TABLE-LIKE MAIN TABLE 6
        # =========================================================

        r10 = {

            row[
                "branch"
            ]:
                row

            for row
            in s0_rankings

            if row[
                "top_pct"
            ] == 10
        }


        paper_rows = []


        for index, result in enumerate(
            s0_results
        ):

            name = (
                result[
                    "branch"
                ]
            )


            if name == "A":

                delta = np.nan

            elif name == "B":

                delta = (
                    result[
                        "AUC"
                    ]
                    -
                    s0_results[
                        0
                    ][
                        "AUC"
                    ]
                )

            else:

                delta = (
                    result[
                        "AUC"
                    ]
                    -
                    s0_results[
                        1
                    ][
                        "AUC"
                    ]
                )


            paper_rows.append(

                dict(

                    branch=name,

                    AUC=result[
                        "AUC"
                    ],

                    delta_AUC=delta,

                    KS=result[
                        "KS"
                    ],

                    Brier=result[
                        "Brier"
                    ],

                    BSS=result[
                        "BSS"
                    ],

                    Precision_at_10=
                        r10[
                            name
                        ][
                            "precision"
                        ],

                    Lift_at_10=
                        r10[
                            name
                        ][
                            "lift"
                        ],

                    Capture_at_10=
                        r10[
                            name
                        ][
                            "capture"
                        ],
                )
            )


        pd.DataFrame(
            paper_rows
        ).to_csv(

            out
            / "S0_table_main.csv",

            index=False,
        )


        # =========================================================
        # S1 COHORT
        #
        # Retain only customers with >=1 post-day60 snapshot.
        # Preserve S0 labels and S0 split assignments.
        # =========================================================

        s1_mask = (

            audit[
                "N_FOLLOWUP_SNAPSHOTS"
            ]
            .to_numpy()
            > 0
        )


        s1_indices = np.where(
            s1_mask
        )[0]


        s1_train = np.array(
            [
                index
                for index
                in tr

                if s1_mask[
                    index
                ]
            ],
            dtype=int,
        )


        s1_test = np.array(
            [
                index
                for index
                in te

                if s1_mask[
                    index
                ]
            ],
            dtype=int,
        )


        # All positives should be retained.
        if (
            y[
                s1_indices
            ].sum()
            != y.sum()
        ):

            raise AssertionError(
                "S1 loại mất positive customer."
            )


        print(
            "\n"
            + "=" * 80
        )

        print(
            "S1 — FOLLOW-UP SENSITIVITY"
        )

        print(
            "=" * 80
        )

        print(
            "S0 N:",
            len(y),
        )

        print(
            "S1 N:",
            len(
                s1_indices
            ),
        )

        print(
            "Excluded:",
            len(y)
            -
            len(
                s1_indices
            ),
        )

        print(
            "S1 positives:",
            int(
                y[
                    s1_indices
                ].sum()
            ),
        )

        print(
            "S1 train:",
            len(
                s1_train
            ),
        )

        print(
            "S1 test:",
            len(
                s1_test
            ),
        )


        # =========================================================
        # S15(a)
        # =========================================================

        s15a = pd.DataFrame(
            [
                dict(
                    Cohort="S0",
                    Customers=len(y),
                    Positive=int(
                        y.sum()
                    ),
                    Train_n=len(tr),
                    Train_positive=int(
                        y[
                            tr
                        ].sum()
                    ),
                    Holdout_n=len(te),
                    Holdout_positive=int(
                        y[
                            te
                        ].sum()
                    ),
                ),

                dict(
                    Cohort="S1",
                    Customers=len(
                        s1_indices
                    ),
                    Positive=int(
                        y[
                            s1_indices
                        ].sum()
                    ),
                    Train_n=len(
                        s1_train
                    ),
                    Train_positive=int(
                        y[
                            s1_train
                        ].sum()
                    ),
                    Holdout_n=len(
                        s1_test
                    ),
                    Holdout_positive=int(
                        y[
                            s1_test
                        ].sum()
                    ),
                ),
            ]
        )


        s15a.to_csv(
            out
            / "Table_S15a_S0_S1_counts.csv",
            index=False,
        )


        # =========================================================
        # S0 MODELS ON S1 HOLDOUT
        #
        # Subset fixed S0 predictions to retained S1 test IDs.
        # =========================================================

        test_position = {

            int(
                global_index
            ):
                position

            for position,
            global_index
            in enumerate(
                te
            )
        }


        s1_test_positions = np.array(

            [
                test_position[
                    int(index)
                ]

                for index
                in s1_test
            ],

            dtype=int,
        )


        s0_on_s1_predictions = {

            branch:
                prediction[
                    s1_test_positions
                ]

            for branch,
            prediction
            in s0_predictions.items()
        }


        y_s1_test = (
            y[
                s1_test
            ]
        )


        # =========================================================
        # S1 REFIT A/B/C
        # =========================================================

        s1_results = []
        s1_predictions = {}


        for name in (
            "A",
            "B",
            "C",
        ):

            (
                result,
                _,
                _,
                scores,
            ) = fit_branch(

                branch_x[
                    name
                ],

                y,

                s1_train,

                s1_test,

                name,

                base_numeric_cols,

                base_categorical_cols,

                run_cv=False,
            )


            s1_results.append(
                result
            )


            s1_predictions[
                name
            ] = scores


        pd.DataFrame(
            s1_results
        ).to_csv(

            out
            / "S1_refit_holdout_metrics.csv",

            index=False,
        )


        # =========================================================
        # S15(b) — AUC + CI
        # =========================================================

        discrimination_rows = []


        evaluations = {

            "S0_full":
                (
                    y_s0_test,
                    s0_predictions,
                    len(tr),
                    len(te),
                ),

            "S0_on_S1_holdout":
                (
                    y_s1_test,
                    s0_on_s1_predictions,
                    len(tr),
                    len(
                        s1_test
                    ),
                ),

            "S1_refit":
                (
                    y_s1_test,
                    s1_predictions,
                    len(
                        s1_train
                    ),
                    len(
                        s1_test
                    ),
                ),
        }


        seed_counter = 0


        for evaluation, (
            labels,
            predictions,
            train_n,
            holdout_n,
        ) in evaluations.items():

            for branch in (
                "A",
                "B",
                "C",
            ):

                auc = (
                    roc_auc_score(
                        labels,
                        predictions[
                            branch
                        ],
                    )
                )


                ci_low, ci_high = (
                    bootstrap_auc_ci(

                        labels,

                        predictions[
                            branch
                        ],

                        BOOTSTRAP_SEED
                        + seed_counter,
                    )
                )


                seed_counter += 1


                discrimination_rows.append(

                    dict(

                        Evaluation=
                            evaluation,

                        Branch=
                            branch,

                        Train_n=
                            train_n,

                        Holdout_n=
                            holdout_n,

                        Holdout_positive=
                            int(
                                labels.sum()
                            ),

                        AUC=
                            auc,

                        CI_lower=
                            ci_low,

                        CI_upper=
                            ci_high,
                    )
                )


        s15b = pd.DataFrame(
            discrimination_rows
        )


        s15b.to_csv(

            out
            / "Table_S15b_discrimination.csv",

            index=False,
        )


        # =========================================================
        # S15(c) — KS / Brier / BSS / Capture@10
        # =========================================================

        additional_rows = []


        for evaluation, (
            labels,
            predictions,
            _,
            _,
        ) in evaluations.items():

            for branch in (
                "A",
                "B",
                "C",
            ):

                result = metrics(
                    labels,
                    predictions[
                        branch
                    ],
                )


                rank10 = ranking(
                    labels,
                    predictions[
                        branch
                    ],
                    branch,
                )[0]


                additional_rows.append(

                    dict(

                        Evaluation=
                            evaluation,

                        Branch=
                            branch,

                        KS=result[
                            "KS"
                        ],

                        Brier=result[
                            "Brier"
                        ],

                        BSS=result[
                            "BSS"
                        ],

                        Capture_at_10=
                            rank10[
                                "capture"
                            ],
                    )
                )


        pd.DataFrame(
            additional_rows
        ).to_csv(

            out
            / "Table_S15c_metrics.csv",

            index=False,
        )


        # =========================================================
        # S16 — CONTRASTS EACH EVALUATION
        # =========================================================

        s16_rows = []


        for eval_index, (
            evaluation,
            (
                labels,
                predictions,
                _,
                _,
            ),
        ) in enumerate(
            evaluations.items()
        ):

            rows, _ = contrasts(

                labels,

                predictions,

                seed=
                    BOOTSTRAP_SEED
                    + 100
                    * eval_index,
            )


            for row in rows:

                row[
                    "Evaluation"
                ] = evaluation


                s16_rows.append(
                    row
                )


        s16 = pd.DataFrame(
            s16_rows
        )


        s16 = s16[
            [
                "Evaluation",
                "contrast",
                "auc_diff",
                "ci_lower",
                "ci_upper",
                "delong_p",
                "bootstrap_valid",
            ]
        ]


        s16.to_csv(

            out
            / "Table_S16_paired_AUC_contrasts.csv",

            index=False,
        )


        # =========================================================
        # S17 — REFITTING EFFECT ON SAME S1 HOLDOUT
        # =========================================================

        s17 = (
            bootstrap_refit_effects(

                y_s1_test,

                s0_on_s1_predictions,

                s1_predictions,
            )
        )


        s17.to_csv(

            out
            / "Table_S17_refitting_effects.csv",

            index=False,
        )


        # =========================================================
        # SAVE S1 PREDICTIONS
        # =========================================================

        s1_prediction_frame = (

            audit.iloc[
                s1_test
            ][
                [
                    "NCODPERS",
                    "ROW_ID",
                    "TARGET_60D",
                ]
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        for branch in (
            "A",
            "B",
            "C",
        ):

            s1_prediction_frame[
                f"S0_SCORE_{branch}"
            ] = (
                s0_on_s1_predictions[
                    branch
                ]
            )


            s1_prediction_frame[
                f"S1_REFIT_SCORE_{branch}"
            ] = (
                s1_predictions[
                    branch
                ]
            )


        s1_prediction_frame.to_csv(

            out
            / "S1_paired_predictions.csv",

            index=False,
        )


        # =========================================================
        # METADATA
        # =========================================================

        metadata = {

            "source_raw":
                str(
                    RAW_FILE
                ),

            "source_sha256":
                sha_file(
                    RAW_FILE
                ),

            "source_bytes":
                RAW_FILE.stat().st_size,

            "run_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "normalization": {

                "stage":
                    "before all feature aggregation",

                "same_rule_for":
                    [
                        "A",
                        "B",
                        "C",
                    ],

                "missing_tokens":
                    [
                        "",
                        "whitespace",
                        "NA",
                        "N/A",
                        "NAN",
                        "NULL",
                        "NONE",
                    ],

                "trim_valid_strings":
                    True,

                "branch_specific_missing_logic":
                    False,
            },

            "target_audit":
                {
                    key:
                        (
                            int(value)
                            if isinstance(
                                value,
                                (
                                    np.integer,
                                )
                            )
                            else value
                        )

                    for key,
                    value
                    in target_audit.items()
                },

            "first_year_filter":
                {

                    key:
                        (
                            str(value)

                            if isinstance(
                                value,
                                (
                                    pd.Timestamp,
                                    datetime,
                                )
                            )

                            else int(value)

                            if isinstance(
                                value,
                                np.integer,
                            )

                            else value
                        )

                    for key,
                    value
                    in source_stats.items()
                },

            "S0": {

                "n":
                    len(y),

                "positives":
                    int(
                        y.sum()
                    ),

                "train":
                    len(tr),

                "holdout":
                    len(te),

                "split_seed":
                    SPLIT_SEED,
            },

            "S1": {

                "rule":
                    "N_FOLLOWUP_SNAPSHOTS > 0",

                "labels_reused_from_S0":
                    True,

                "split_assignment_reused":
                    True,

                "n":
                    len(
                        s1_indices
                    ),

                "positives":
                    int(
                        y[
                            s1_indices
                        ].sum()
                    ),

                "train":
                    len(
                        s1_train
                    ),

                "holdout":
                    len(
                        s1_test
                    ),
            },

            "protocol":
                (
                    "Filter relationship-start <= snapshot "
                    "< relationship-start + 1 calendar year; "
                    "A uses snapshots through day 60 inclusive; "
                    "target is card status after day 60; "
                    "B positive cutoff is first positive post-day60 "
                    "snapshot and negative cutoff is min(first-year "
                    "endpoint, administrative endpoint), both exclusive; "
                    "C equals B plus cutoff tenure."
                ),

            "model":
                "XGBoost no sampling; separately fitted A/B/C",

            "xgb_params":
                XGB_PARAMS,

            "shared_schema":
                schema,

            "cv_seed":
                CV_SEED,

            "bootstrap_seed":
                BOOTSTRAP_SEED,

            "bootstrap_n":
                N_BOOTSTRAP,

            "environment": {

                "python":
                    platform.python_version(),

                "numpy":
                    np.__version__,

                "pandas":
                    pd.__version__,

                "scikit_learn":
                    sklearn.__version__,

                "xgboost":
                    xgboost.__version__,

                "duckdb":
                    duckdb.__version__,

                "scipy":
                    scipy.__version__,
            },
        }


        (
            out
            / "run_metadata.json"
        ).write_text(

            json.dumps(
                metadata,
                indent=2,
                ensure_ascii=False,
                default=str,
            ),

            encoding="utf-8",
        )


        # =========================================================
        # PRINT FINAL TABLES
        # =========================================================

        print(
            "\n\n"
            + "=" * 100
        )

        print(
            "S0 HOLDOUT METRICS"
        )

        print(
            "=" * 100
        )

        print(

            pd.DataFrame(
                s0_results
            )[
                [
                    "branch",
                    "AUC",
                    "KS",
                    "Recall",
                    "Precision",
                    "F1",
                    "Accuracy",
                    "Brier",
                    "BSS",
                ]
            ]
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "S0 PAIRED AUC CONTRASTS"
        )

        print(
            "=" * 100
        )

        print(

            pd.DataFrame(
                s0_contrast_rows
            )
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "TABLE S15(a)"
        )

        print(
            "=" * 100
        )

        print(
            s15a.to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "TABLE S15(b)"
        )

        print(
            "=" * 100
        )

        print(

            s15b
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "TABLE S15(c)"
        )

        print(
            "=" * 100
        )

        print(

            pd.DataFrame(
                additional_rows
            )
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "TABLE S16 — PAIRED AUC"
        )

        print(
            "=" * 100
        )

        print(

            s16
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "TABLE S17 — REFIT EFFECT"
        )

        print(
            "=" * 100
        )

        print(

            s17
            .round(6)
            .to_string(
                index=False
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            "RUN COMPLETED"
        )

        print(
            "=" * 100
        )

        print(
            "Saved to:",
            out,
        )


    finally:

        shutil.rmtree(
            work,
            ignore_errors=True,
        )


# =====================================================================
# 19. RUN
# =====================================================================

if __name__ == "__main__":

    main()

Mounted at /content/drive
SANTANDER A/B/C — UNIFIED NORMALIZATION
Raw: /content/drive/MyDrive/Santander_CreditCard.csv
Output: /content/drive/MyDrive/Santander_ABC_UnifiedNormalization/run_20261012T082702859015Z


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


TARGET AUDIT
n_rows              13619575
n_missing_target           0
n_invalid_target           0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Customers with multiple fecha_alta: 17


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


S0 FIRST-YEAR COHORT
N           : 153,399
Positive    : 916
Train       : 122,719
Holdout     : 30,680
No post60 snapshot: 14332

SHARED FEATURE SCHEMA
Numeric: ['AGE', 'IND_NUEVO', 'ANTIGUEDAD', 'INDREL', 'INDREL_1MES', 'TIPODOM', 'COD_PROV', 'IND_ACTIVIDAD_CLIENTE', 'RENTA']
Categorical: ['IND_EMPLEADO', 'PAIS_RESIDENCIA', 'SEXO', 'TIPREL_1MES', 'INDRESI', 'INDEXT', 'CONYUEMP', 'INDFALL', 'NOMPROV', 'SEGMENTO', 'CANAL_ENTRADA']

S0 — CANONICAL A/B/C
A CV 1/5: AUC=0.870810
A CV 2/5: AUC=0.869443
A CV 3/5: AUC=0.867107
A CV 4/5: AUC=0.863839
A CV 5/5: AUC=0.871616
A TEST: AUC=0.885617 | KS=0.647980 | Brier=0.005831
B CV 1/5: AUC=0.960896
B CV 2/5: AUC=0.970723
B CV 3/5: AUC=0.977977
B CV 4/5: AUC=0.958894
B CV 5/5: AUC=0.959768
B TEST: AUC=0.972259 | KS=0.819746 | Brier=0.003353
C CV 1/5: AUC=0.997009
C CV 2/5: AUC=0.997544
C CV 3/5: AUC=0.997797
C CV 4/5: AUC=0.995596
C CV 5/5: AUC=0.990516
C TEST: AUC=0.998651 | KS=0.968654 | Brier=0.001296

S1 — FOLLOW-UP SENSITIVITY
S0 N: 153399


#B A / B / B-sym

In [ ]:
# ======================================================================
# ELAPS — SANTANDER A / B / B-SYM
# S0 CANONICAL + R50 FOUR LEARNERS + S1 SENSITIVITY
#
# Version: unified missing normalization + symmetric negative cutoffs
#
# PRIMARY BRANCHES
# ----------------------------------------------------------------------
# A:
#   Fixed landmark at relationship_start + 60 days.
#
# B:
#   Positive -> cutoff = first credit-card event after day 60
#   Negative -> cutoff = min(first-year end, administrative end)
#
# B-sym:
#   Positive -> same actual event cutoff as B
#   Negative -> pseudo duration sampled from
#               {event_date - relationship_start}
#               among POSITIVE TRAINING customers of the SAME split.
#
#   pseudo_cutoff =
#       min(relationship_start + sampled_positive_duration, T_end)
#
# where
#   T_end = min(first-year endpoint, administrative endpoint)
#
# IMPORTANT
# ----------------------------------------------------------------------
# - Same customers / labels / split for A, B and B-sym.
# - Missing strings normalized ONCE before any aggregation.
# - Feature preprocessing fit TRAIN ONLY.
# - LR additionally uses StandardScaler fitted TRAIN ONLY.
# - No resampling.
# - S0: XGB canonical + 1,000 paired bootstrap.
# - R50: 4 learners × A/B/B-sym × 50 split/refit seeds.
# - S1: retain N_FOLLOWUP_SNAPSHOTS > 0, preserve S0 seed-42
#       split membership, refit A/B/B-sym.
# ======================================================================


# ======================================================================
# 0. INSTALL IF NEEDED
# ======================================================================

import importlib.util
import subprocess
import sys

PACKAGES = [
    ("duckdb", "duckdb"),
    ("xgboost", "xgboost"),
    ("lightgbm", "lightgbm"),
    ("sklearn", "scikit-learn"),
    ("scipy", "scipy"),
]

for module, package in PACKAGES:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )


# ======================================================================
# 1. IMPORTS
# ======================================================================

from datetime import datetime, timezone
from pathlib import Path
from hashlib import sha256

import gc
import json
import platform
import shutil
import warnings

warnings.filterwarnings("ignore")

import duckdb
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost
import lightgbm

from scipy.stats import kendalltau

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ======================================================================
# 2. GOOGLE DRIVE
# ======================================================================

try:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

except ImportError:
    pass


# ======================================================================
# 3. PATHS
# ======================================================================

ROOT = Path("/content/drive/MyDrive")

RAW_FILE = ROOT / "Santander_CreditCard.csv"

OUTPUT_PARENT = (
    ROOT
    / "Santander_AB_Bsym_S0_R50_S1"
)

WORK_PARENT = Path("/content")


# ======================================================================
# 4. GLOBAL CONFIG
# ======================================================================

CANONICAL_SPLIT_SEED = 42

# 50 seeds = 42,...,91
R50_SEEDS = list(range(42, 92))

assert len(R50_SEEDS) == 50


# B-sym pseudo-duration randomization
BSYM_SEED_BASE = 20261012


# Paired bootstrap
N_BOOTSTRAP = 1000

BOOTSTRAP_SEED_S0 = 2026101201
BOOTSTRAP_SEED_S1 = 2026101202


# Number of threads
N_JOBS = 2


# ======================================================================
# 5. SANTANDER FEATURES
# ======================================================================

FEATURES = [
    "IND_EMPLEADO",
    "PAIS_RESIDENCIA",
    "SEXO",
    "AGE",
    "IND_NUEVO",
    "ANTIGUEDAD",
    "INDREL",
    "INDREL_1MES",
    "TIPREL_1MES",
    "INDRESI",
    "INDEXT",
    "CONYUEMP",
    "INDFALL",
    "TIPODOM",
    "COD_PROV",
    "NOMPROV",
    "IND_ACTIVIDAD_CLIENTE",
    "RENTA",
    "SEGMENTO",
    "CANAL_ENTRADA",
]

assert len(FEATURES) == 20
assert len(set(FEATURES)) == 20


# ======================================================================
# 6. COMMON MISSING NORMALIZATION
# ======================================================================

MISSING_SQL_VALUES = (
    "'', "
    "'NA', "
    "'N/A', "
    "'NAN', "
    "'NULL', "
    "'NONE'"
)

MISSING_PYTHON_VALUES = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NAN",
    "nan",
    "NULL",
    "null",
    "None",
    "none",
]


def clean_sql_expression(expression):
    """
    Same normalization for ALL branches BEFORE aggregation.
    """

    return (
        "CASE "
        f"WHEN {expression} IS NULL THEN NULL "
        f"WHEN UPPER(TRIM({expression})) "
        f"IN ({MISSING_SQL_VALUES}) THEN NULL "
        f"ELSE TRIM({expression}) "
        "END"
    )


def normalize_missing_frame(frame):

    return (
        frame.copy()
        .replace(
            MISSING_PYTHON_VALUES,
            np.nan,
        )
    )


# ======================================================================
# 7. HELPERS
# ======================================================================

def sql_string(value):

    return (
        "'"
        + str(value).replace("'", "''")
        + "'"
    )


def sha_file(path):

    h = sha256()

    with Path(path).open("rb") as f:

        for piece in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(piece)

    return h.hexdigest()


# ======================================================================
# 8. BUILD RAW / FIRST-YEAR / A / B
# ======================================================================

def build_base_tables(
    con,
    out,
):

    if not RAW_FILE.is_file():

        raise FileNotFoundError(
            f"Không tìm thấy:\n{RAW_FILE}"
        )


    source = (
        f"read_csv("
        f"{sql_string(RAW_FILE)}, "
        f"header=true, "
        f"all_varchar=true, "
        f"sample_size=10000, "
        f"ignore_errors=true"
        f")"
    )


    # ------------------------------------------------------------------
    # 8.1 Column audit
    # ------------------------------------------------------------------

    columns = set(

        con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM {source}
            LIMIT 1
            """
        )
        .fetchdf()["column_name"]
        .str.lower()
    )


    required = {
        "ncodpers",
        "fecha_dato",
        "fecha_alta",
        "ind_tjcr_fin_ult1",
    }


    missing_required = required - columns

    if missing_required:

        raise ValueError(
            "Thiếu cột bắt buộc: "
            + str(
                sorted(
                    missing_required
                )
            )
        )


    missing_features = [
        f
        for f in FEATURES
        if f.lower() not in columns
    ]


    if missing_features:

        raise ValueError(
            "Thiếu predictor Santander: "
            + ", ".join(
                missing_features
            )
        )


    # ------------------------------------------------------------------
    # 8.2 Unified feature normalization
    # ------------------------------------------------------------------

    feature_sql = ",\n".join(

        f"""
        {clean_sql_expression(
            f's."{feature.lower()}"'
        )}
        AS "{feature}"
        """

        for feature in FEATURES
    )


    target_sql = clean_sql_expression(
        "s.ind_tjcr_fin_ult1"
    )


    # ------------------------------------------------------------------
    # 8.3 Raw normalized data
    # ------------------------------------------------------------------

    con.execute(
        f"""
        CREATE OR REPLACE TABLE raw_data AS

        SELECT

            TRY_CAST(
                NULLIF(TRIM(s.ncodpers),'')
                AS BIGINT
            ) AS ncodpers,

            TRY_CAST(
                NULLIF(TRIM(s.fecha_dato),'')
                AS DATE
            ) AS fecha_dato,

            TRY_CAST(
                NULLIF(TRIM(s.fecha_alta),'')
                AS DATE
            ) AS fecha_alta,

            TRY_CAST(
                {target_sql}
                AS DOUBLE
            ) AS card_status,

            {feature_sql}

        FROM {source} AS s

        WHERE

            TRY_CAST(
                NULLIF(TRIM(s.ncodpers),'')
                AS BIGINT
            ) IS NOT NULL

            AND

            TRY_CAST(
                NULLIF(TRIM(s.fecha_dato),'')
                AS DATE
            ) IS NOT NULL

            AND

            TRY_CAST(
                NULLIF(TRIM(s.fecha_alta),'')
                AS DATE
            ) IS NOT NULL
        """
    )


    # ------------------------------------------------------------------
    # 8.4 Target audit
    # ------------------------------------------------------------------

    target_audit = (

        con.execute(
            """
            SELECT

                COUNT(*) AS n_rows,

                COUNT(*)
                FILTER (
                    WHERE card_status IS NULL
                )
                AS n_missing_target,

                COUNT(*)
                FILTER (
                    WHERE card_status IS NOT NULL
                    AND card_status NOT IN (0,1)
                )
                AS n_invalid_target

            FROM raw_data
            """
        )
        .fetchdf()
        .iloc[0]
    )


    print("\n" + "=" * 80)
    print("TARGET AUDIT")
    print("=" * 80)

    print(
        target_audit.to_string()
    )


    if int(
        target_audit[
            "n_missing_target"
        ]
    ) != 0:

        raise ValueError(
            "Có target missing; "
            "không được tự gán missing thành 0."
        )


    if int(
        target_audit[
            "n_invalid_target"
        ]
    ) != 0:

        raise ValueError(
            "Target có giá trị ngoài {0,1}."
        )


    # ------------------------------------------------------------------
    # 8.5 Relationship start audit
    # ------------------------------------------------------------------

    multiple_start = (
        con.execute(
            """
            SELECT COUNT(*)

            FROM (

                SELECT
                    ncodpers,
                    COUNT(DISTINCT fecha_alta)
                        AS n_dates

                FROM raw_data

                GROUP BY ncodpers

                HAVING
                    COUNT(DISTINCT fecha_alta) > 1
            )
            """
        )
        .fetchone()[0]
    )


    print(
        "\nCustomers with multiple fecha_alta:",
        multiple_start,
    )


    # ------------------------------------------------------------------
    # 8.6 Canonical relationship start
    # ------------------------------------------------------------------

    con.execute(
        """
        CREATE OR REPLACE TABLE normalized AS

        SELECT

            * EXCLUDE(
                fecha_alta,
                card_status
            ),

            MIN(fecha_alta)
            OVER (
                PARTITION BY ncodpers
            )
            AS fecha_alta,

            CASE
                WHEN card_status > 0
                THEN 1
                ELSE 0
            END
            AS has_card

        FROM raw_data
        """
    )


    # ------------------------------------------------------------------
    # 8.7 Source statistics
    # ------------------------------------------------------------------

    stats = (

        con.execute(
            """
            SELECT

                COUNT(*) AS valid_source_rows,

                COUNT(*)
                FILTER (
                    WHERE fecha_dato < fecha_alta
                )
                AS before_relationship_rows,

                COUNT(*)
                FILTER (
                    WHERE fecha_dato
                        >= fecha_alta + INTERVAL '1 year'
                )
                AS rows_at_or_after_first_year,

                COUNT(*)
                FILTER (
                    WHERE fecha_dato >= fecha_alta
                    AND fecha_dato
                        < fecha_alta + INTERVAL '1 year'
                )
                AS first_year_rows,

                MIN(fecha_dato)
                    AS source_first_date,

                MAX(fecha_dato)
                    AS source_last_date

            FROM normalized
            """
        )
        .fetchdf()
        .iloc[0]
        .to_dict()
    )


    admin_end = (
        pd.Timestamp(
            stats[
                "source_last_date"
            ]
        )
        + pd.Timedelta(days=1)
    )


    # ------------------------------------------------------------------
    # 8.8 Common first-year snapshot table
    # ------------------------------------------------------------------

    con.execute(
        """
        CREATE OR REPLACE TABLE first_year_data AS

        SELECT *

        FROM normalized

        WHERE

            fecha_dato >= fecha_alta

            AND

            fecha_dato
                < fecha_alta + INTERVAL '1 year'
        """
    )


    duplicate = con.execute(
        """
        SELECT
            ncodpers,
            fecha_dato,
            COUNT(*) AS n

        FROM first_year_data

        GROUP BY 1,2

        HAVING COUNT(*) > 1

        LIMIT 1
        """
    ).fetchone()


    if duplicate:

        raise ValueError(
            f"Duplicate customer/date snapshot: "
            f"{duplicate}"
        )


    # ------------------------------------------------------------------
    # 8.9 Common aggregation expressions
    # ------------------------------------------------------------------

    last_features = FEATURES[:-1]


    latest_expr = ",\n".join(

        f"""
        ARG_MAX(
            "{feature}",
            fecha_dato
        )
        FILTER (
            WHERE "{feature}" IS NOT NULL
        )
        AS "{feature}"
        """

        for feature in last_features
    )


    latest_expr += """
    ,
    ARG_MIN(
        "CANAL_ENTRADA",
        fecha_dato
    )
    FILTER (
        WHERE "CANAL_ENTRADA" IS NOT NULL
    )
    AS "CANAL_ENTRADA"
    """


    # ------------------------------------------------------------------
    # 8.10 Branch A
    # ------------------------------------------------------------------

    con.execute(
        f"""
        CREATE OR REPLACE TABLE a_window AS

        SELECT

            ncodpers,

            fecha_alta,

            CAST(
                fecha_alta + INTERVAL '60 days'
                AS DATE
            )
            AS cutoff_date,

            CAST(
                fecha_alta + INTERVAL '1 year'
                AS DATE
            )
            AS year_end_exclusive,

            MAX(fecha_dato)
                AS last_feature_snapshot,

            COUNT(*)
                AS n_feature_snapshots,

            MAX(has_card)
                AS adopted_within_60d,

            {latest_expr}

        FROM first_year_data

        WHERE

            fecha_dato >= fecha_alta

            AND

            fecha_dato
                <= fecha_alta + INTERVAL '60 days'

        GROUP BY
            ncodpers,
            fecha_alta
        """
    )


    # ------------------------------------------------------------------
    # 8.11 Post-day-60 outcome
    # ------------------------------------------------------------------

    con.execute(
        """
        CREATE OR REPLACE TABLE after_60 AS

        SELECT

            ncodpers,

            MAX(has_card)::INTEGER
                AS target_60d,

            MIN(fecha_dato)
            FILTER (
                WHERE has_card = 1
            )
            AS first_card_date_after_cutoff,

            MAX(fecha_dato)
                AS last_available_snapshot,

            COUNT(*)
                AS n_followup_snapshots

        FROM first_year_data

        WHERE

            fecha_dato
                > fecha_alta + INTERVAL '60 days'

        GROUP BY ncodpers
        """
    )


    feature_select_A = ", ".join(
        f'a."{c}"'
        for c in FEATURES
    )


    con.execute(
        f"""
        CREATE OR REPLACE TABLE a_cohort AS

        SELECT

            a.ncodpers
                AS "NCODPERS",

            a.fecha_alta
                AS "FECHA_ALTA",

            a.cutoff_date
                AS "A_CUTOFF",

            a.year_end_exclusive
                AS "YEAR_END_EXCLUSIVE",

            a.last_feature_snapshot
                AS "A_LAST_FEATURE_DATE",

            a.n_feature_snapshots
                AS "A_N_FEATURE_SNAPSHOTS",

            t.first_card_date_after_cutoff
                AS "EVENT_DATE",

            t.last_available_snapshot
                AS "LAST_AVAILABLE_SNAPSHOT",

            COALESCE(
                t.n_followup_snapshots,
                0
            )
            AS "N_FOLLOWUP_SNAPSHOTS",

            {feature_select_A},

            COALESCE(
                t.target_60d,
                0
            )::INTEGER
            AS "TARGET_60D"

        FROM a_window a

        LEFT JOIN after_60 t
        USING (ncodpers)

        WHERE
            a.adopted_within_60d = 0
        """
    )


    audit = (

        con.execute(
            """
            SELECT *
            FROM a_cohort
            ORDER BY "NCODPERS"
            """
        )
        .fetchdf()
    )


    # ------------------------------------------------------------------
    # 8.12 A cohort audit
    # ------------------------------------------------------------------

    if audit.empty:

        raise ValueError(
            "S0 cohort rỗng."
        )


    if audit[
        "TARGET_60D"
    ].nunique() != 2:

        raise ValueError(
            "Target S0 chỉ còn một class."
        )


    if audit[
        "NCODPERS"
    ].duplicated().any():

        raise AssertionError(
            "NCODPERS duplicated."
        )


    positive = (
        audit[
            "TARGET_60D"
        ].eq(1)
    )


    if not (
        audit.loc[
            positive,
            "EVENT_DATE"
        ]
        >
        audit.loc[
            positive,
            "A_CUTOFF"
        ]
    ).all():

        raise AssertionError(
            "Positive event <= day-60 cutoff."
        )


    if not (
        audit.loc[
            positive,
            "EVENT_DATE"
        ]
        <
        audit.loc[
            positive,
            "YEAR_END_EXCLUSIVE"
        ]
    ).all():

        raise AssertionError(
            "Positive event outside first year."
        )


    # ------------------------------------------------------------------
    # 8.13 Branch B actual cutoff
    # ------------------------------------------------------------------

    latest_expr_B = ",\n".join(

        f"""
        ARG_MAX(
            s."{feature}",
            s.fecha_dato
        )
        FILTER (
            WHERE s."{feature}" IS NOT NULL
        )
        AS "{feature}"
        """

        for feature in last_features
    )


    latest_expr_B += """
    ,
    ARG_MIN(
        s."CANAL_ENTRADA",
        s.fecha_dato
    )
    FILTER (
        WHERE s."CANAL_ENTRADA" IS NOT NULL
    )
    AS "CANAL_ENTRADA"
    """


    con.execute(
        f"""
        CREATE OR REPLACE TABLE b_cohort AS

        WITH cutoffs AS (

            SELECT

                a."NCODPERS"
                    AS ncodpers,

                CASE

                    WHEN
                        a."TARGET_60D" = 1

                    THEN
                        a."EVENT_DATE"

                    ELSE
                        LEAST(
                            a."YEAR_END_EXCLUSIVE",
                            DATE {
                                sql_string(
                                    admin_end.date()
                                )
                            }
                        )

                END
                AS b_cutoff

            FROM a_cohort a
        )


        SELECT

            c.ncodpers
                AS "NCODPERS",

            c.b_cutoff
                AS "B_CUTOFF",

            MAX(s.fecha_dato)
                AS "B_LAST_FEATURE_DATE",

            COUNT(s.fecha_dato)
                AS "B_N_FEATURE_SNAPSHOTS",

            {latest_expr_B}

        FROM cutoffs c

        LEFT JOIN first_year_data s

            ON
                s.ncodpers = c.ncodpers

                AND

                s.fecha_dato < c.b_cutoff

        GROUP BY

            c.ncodpers,
            c.b_cutoff

        ORDER BY c.ncodpers
        """
    )


    b = (

        con.execute(
            """
            SELECT *
            FROM b_cohort
            ORDER BY "NCODPERS"
            """
        )
        .fetchdf()
    )


    if not np.array_equal(
        audit[
            "NCODPERS"
        ].to_numpy(),
        b[
            "NCODPERS"
        ].to_numpy(),
    ):

        raise AssertionError(
            "A and B customer order mismatch."
        )


    print("\n" + "=" * 80)
    print("S0 BASE COHORT")
    print("=" * 80)

    print(
        f"N                    : {len(audit):,}"
    )

    print(
        f"Positive             : "
        f"{int(audit.TARGET_60D.sum()):,}"
    )

    print(
        f"No post60 snapshot   : "
        f"{int(audit.N_FOLLOWUP_SNAPSHOTS.eq(0).sum()):,}"
    )


    stats[
        "multiple_fecha_alta_customers"
    ] = int(
        multiple_start
    )


    pd.DataFrame(
        [stats]
    ).to_csv(

        out
        / "source_filter_summary.csv",

        index=False,
    )


    return (
        audit,
        b,
        admin_end,
        stats,
        target_audit.to_dict(),
    )


# ======================================================================
# 9. BUILD B-SYM FOR ONE SPLIT
# ======================================================================

def build_bsym_matrix(
    con,
    audit,
    train_idx,
    split_seed,
    admin_end,
    save_cutoff_path=None,
):

    """
    Positive:
        B-sym cutoff = actual event date.

    Negative:
        draw duration from POSITIVE TRAIN customers:
            event_date - fecha_alta

        pseudo cutoff =
            min(fecha_alta + sampled_duration, T_end)

        where
            T_end =
                min(first-year endpoint, administrative endpoint)

    No holdout-positive event-duration distribution is used.
    """


    train_idx = np.asarray(
        train_idx,
        dtype=int,
    )


    train_positive_mask = (
        audit.iloc[
            train_idx
        ][
            "TARGET_60D"
        ]
        .to_numpy()
        == 1
    )


    train_positive_idx = (
        train_idx[
            train_positive_mask
        ]
    )


    durations = (

        audit.iloc[
            train_positive_idx
        ][
            "EVENT_DATE"
        ].reset_index(drop=True)

        -

        audit.iloc[
            train_positive_idx
        ][
            "FECHA_ALTA"
        ].reset_index(drop=True)

    ).dt.days.to_numpy()


    durations = durations[
        np.isfinite(
            durations
        )
    ].astype(int)


    if len(durations) == 0:

        raise ValueError(
            "Không có positive duration "
            "trong training set."
        )


    if np.min(durations) <= 60:

        raise AssertionError(
            "Có positive duration <= 60."
        )


    rng = np.random.default_rng(
        BSYM_SEED_BASE
        + int(split_seed)
    )


    n = len(audit)


    is_positive = (
        audit[
            "TARGET_60D"
        ].to_numpy()
        == 1
    )


    sampled_days = np.full(
        n,
        np.nan,
    )


    negative_idx = np.where(
        ~is_positive
    )[0]


    sampled_days[
        negative_idx
    ] = rng.choice(

        durations,

        size=len(
            negative_idx
        ),

        replace=True,
    )


    # Positive duration = actual event duration
    positive_duration = (

        audit.loc[
            is_positive,
            "EVENT_DATE"
        ]

        -

        audit.loc[
            is_positive,
            "FECHA_ALTA"
        ]

    ).dt.days.to_numpy()


    sampled_days[
        is_positive
    ] = positive_duration


    # --------------------------------------------------------------
    # Individual T_end
    # --------------------------------------------------------------

    year_end = pd.to_datetime(
        audit[
            "YEAR_END_EXCLUSIVE"
        ]
    )


    admin_scalar = pd.Timestamp(
        admin_end
    )


    t_end = year_end.copy()

    t_end = t_end.where(
        t_end <= admin_scalar,
        admin_scalar,
    )


    proposed_cutoff = (

        pd.to_datetime(
            audit[
                "FECHA_ALTA"
            ]
        )

        +

        pd.to_timedelta(
            sampled_days,
            unit="D",
        )
    )


    cutoff = proposed_cutoff.copy()


    negative_mask = (
        ~is_positive
    )


    cutoff.loc[
        negative_mask
    ] = np.minimum(

        proposed_cutoff.loc[
            negative_mask
        ].to_numpy(
            dtype="datetime64[ns]"
        ),

        t_end.loc[
            negative_mask
        ].to_numpy(
            dtype="datetime64[ns]"
        ),
    )


    # Positive cutoff exactly actual event date
    cutoff.loc[
        is_positive
    ] = pd.to_datetime(

        audit.loc[
            is_positive,
            "EVENT_DATE"
        ]
    )


    cutoff_df = pd.DataFrame(
        {
            "row_pos":
                np.arange(
                    n,
                    dtype=np.int64,
                ),

            "ncodpers":
                audit[
                    "NCODPERS"
                ].to_numpy(),

            "bsym_cutoff":
                pd.to_datetime(
                    cutoff
                ),

            "sampled_duration_days":
                sampled_days.astype(int),

            "target":
                audit[
                    "TARGET_60D"
                ].to_numpy(
                    dtype=int
                ),

            "t_end":
                pd.to_datetime(
                    t_end
                ),
        }
    )


    cutoff_df[
        "capped_at_Tend"
    ] = (

        (
            cutoff_df[
                "target"
            ] == 0
        )

        &

        (
            cutoff_df[
                "bsym_cutoff"
            ]
            <
            proposed_cutoff
        )
    )


    if cutoff_df[
        "bsym_cutoff"
    ].isna().any():

        raise AssertionError(
            "B-sym has missing cutoff."
        )


    # Save canonical cutoff audit if requested.
    if save_cutoff_path is not None:

        cutoff_df.to_csv(
            save_cutoff_path,
            index=False,
        )


    # --------------------------------------------------------------
    # Register cutoffs in DuckDB
    # --------------------------------------------------------------

    con.register(
        "tmp_bsym_cutoffs",
        cutoff_df[
            [
                "row_pos",
                "ncodpers",
                "bsym_cutoff",
            ]
        ],
    )


    last_features = (
        FEATURES[:-1]
    )


    bsym_expr = ",\n".join(

        f"""
        ARG_MAX(
            s."{feature}",
            s.fecha_dato
        )
        FILTER (
            WHERE s."{feature}" IS NOT NULL
        )
        AS "{feature}"
        """

        for feature
        in last_features
    )


    bsym_expr += """
    ,
    ARG_MIN(
        s."CANAL_ENTRADA",
        s.fecha_dato
    )
    FILTER (
        WHERE s."CANAL_ENTRADA" IS NOT NULL
    )
    AS "CANAL_ENTRADA"
    """


    bsym = con.execute(
        f"""
        SELECT

            c.row_pos,

            c.ncodpers
                AS "NCODPERS",

            c.bsym_cutoff
                AS "BSYM_CUTOFF",

            MAX(s.fecha_dato)
                AS "BSYM_LAST_FEATURE_DATE",

            COUNT(s.fecha_dato)
                AS "BSYM_N_FEATURE_SNAPSHOTS",

            {bsym_expr}

        FROM tmp_bsym_cutoffs c

        LEFT JOIN first_year_data s

            ON
                s.ncodpers = c.ncodpers

                AND

                s.fecha_dato
                    < c.bsym_cutoff

        GROUP BY

            c.row_pos,
            c.ncodpers,
            c.bsym_cutoff

        ORDER BY
            c.row_pos
        """
    ).fetchdf()


    try:
        con.unregister(
            "tmp_bsym_cutoffs"
        )
    except Exception:
        pass


    if len(
        bsym
    ) != len(
        audit
    ):

        raise AssertionError(
            "B-sym changed cohort size."
        )


    if not np.array_equal(
        bsym[
            "NCODPERS"
        ].to_numpy(),
        audit[
            "NCODPERS"
        ].to_numpy(),
    ):

        raise AssertionError(
            "B-sym changed customer order."
        )


    info = {
        "split_seed":
            int(split_seed),

        "positive_train_durations_n":
            int(len(durations)),

        "positive_duration_mean":
            float(
                np.mean(
                    durations
                )
            ),

        "positive_duration_median":
            float(
                np.median(
                    durations
                )
            ),

        "positive_duration_min":
            int(
                np.min(
                    durations
                )
            ),

        "positive_duration_max":
            int(
                np.max(
                    durations
                )
            ),

        "negative_cutoffs_capped":
            int(
                cutoff_df[
                    "capped_at_Tend"
                ].sum()
            ),
    }


    return (
        bsym[
            FEATURES
        ].copy(),
        cutoff_df,
        info,
    )


# ======================================================================
# 10. SHARED FEATURE TYPE SCHEMA
# ======================================================================

def infer_schema_from_A_train(
    A,
    train_idx,
):

    train = normalize_missing_frame(

        A.iloc[
            train_idx
        ]
    )


    numeric = []
    categorical = []


    for column in train.columns:

        converted = pd.to_numeric(
            train[
                column
            ],
            errors="coerce",
        )


        denominator = int(
            train[
                column
            ]
            .notna()
            .sum()
        )


        numeric_count = int(
            converted
            .notna()
            .sum()
        )


        ratio = (
            numeric_count
            / denominator

            if denominator > 0

            else 0.0
        )


        if ratio >= 0.80:

            numeric.append(
                column
            )

        else:

            categorical.append(
                column
            )


    return (
        numeric,
        categorical,
    )


# ======================================================================
# 11. PREPROCESS TRAIN / HOLDOUT
# ======================================================================

def preprocess_branch(
    X,
    train_idx,
    test_idx,
    numeric_cols,
    categorical_cols,
    scale=False,
):

    train = normalize_missing_frame(

        X.iloc[
            train_idx
        ].reset_index(
            drop=True
        )
    )


    test = normalize_missing_frame(

        X.iloc[
            test_idx
        ].reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------------
    # Numeric
    # --------------------------------------------------------------

    for col in numeric_cols:

        train[
            col
        ] = pd.to_numeric(
            train[
                col
            ],
            errors="coerce",
        )

        test[
            col
        ] = pd.to_numeric(
            test[
                col
            ],
            errors="coerce",
        )


    # --------------------------------------------------------------
    # Categorical
    # --------------------------------------------------------------

    if categorical_cols:

        for col in categorical_cols:

            train[
                col
            ] = (
                train[
                    col
                ]
                .astype("string")
                .fillna("UNKNOWN")
            )


            test[
                col
            ] = (
                test[
                    col
                ]
                .astype("string")
                .fillna("UNKNOWN")
            )


        encoder = OrdinalEncoder(

            handle_unknown=
                "use_encoded_value",

            unknown_value=-1,

            encoded_missing_value=-1,

            dtype=np.float32,
        )


        train[
            categorical_cols
        ] = encoder.fit_transform(

            train[
                categorical_cols
            ]
        )


        test[
            categorical_cols
        ] = encoder.transform(

            test[
                categorical_cols
            ]
        )


    # --------------------------------------------------------------
    # Final numeric matrix
    # --------------------------------------------------------------

    for col in train.columns:

        train[
            col
        ] = pd.to_numeric(
            train[
                col
            ],
            errors="coerce",
        )

        test[
            col
        ] = pd.to_numeric(
            test[
                col
            ],
            errors="coerce",
        )


    # --------------------------------------------------------------
    # Median imputation — TRAIN ONLY
    # --------------------------------------------------------------

    try:

        imputer = SimpleImputer(
            strategy="median",
            keep_empty_features=True,
        )

    except TypeError:

        imputer = SimpleImputer(
            strategy="median"
        )


    X_train = (
        imputer.fit_transform(
            train
        )
        .astype(
            np.float32
        )
    )


    X_test = (
        imputer.transform(
            test
        )
        .astype(
            np.float32
        )
    )


    # --------------------------------------------------------------
    # StandardScaler only for LR
    # --------------------------------------------------------------

    if scale:

        scaler = StandardScaler()

        X_train = (
            scaler.fit_transform(
                X_train
            )
            .astype(
                np.float32
            )
        )


        X_test = (
            scaler.transform(
                X_test
            )
            .astype(
                np.float32
            )
        )


    return (
        X_train,
        X_test,
    )


# ======================================================================
# 12. MODEL FACTORY
# ======================================================================

def make_model(
    learner,
    seed,
):

    if learner == "LR":

        return LogisticRegression(

            max_iter=2000,

            solver="lbfgs",

            class_weight="balanced",

            random_state=seed,
        )


    if learner == "RF":

        return RandomForestClassifier(

            n_estimators=200,

            random_state=seed,

            n_jobs=N_JOBS,
        )


    if learner == "LightGBM":

        return LGBMClassifier(

            n_estimators=500,

            learning_rate=0.05,

            num_leaves=31,

            max_depth=6,

            subsample=0.8,

            colsample_bytree=0.8,

            random_state=seed,

            n_jobs=N_JOBS,

            verbosity=-1,
        )


    if learner == "XGBoost":

        return XGBClassifier(

            n_estimators=300,

            max_depth=6,

            learning_rate=0.05,

            subsample=0.8,

            colsample_bytree=0.8,

            objective="binary:logistic",

            eval_metric="logloss",

            tree_method="hist",

            max_bin=256,

            random_state=seed,

            n_jobs=N_JOBS,
        )


    raise ValueError(
        f"Unknown learner: {learner}"
    )


# ======================================================================
# 13. METRICS
# ======================================================================

def binary_metrics(
    y,
    scores,
):

    y = np.asarray(
        y,
        dtype=int,
    )

    scores = np.asarray(
        scores,
        dtype=float,
    )


    prediction = (
        scores >= 0.5
    ).astype(int)


    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            prediction,
            labels=[0,1],
        )
        .ravel()
    )


    fpr, tpr, _ = roc_curve(
        y,
        scores,
    )


    prevalence = float(
        np.mean(
            y
        )
    )


    brier = (
        brier_score_loss(
            y,
            scores,
        )
    )


    baseline_brier = (
        prevalence
        *
        (
            1
            - prevalence
        )
    )


    bss = (

        1
        -
        brier
        /
        baseline_brier

        if baseline_brier > 0

        else np.nan
    )


    return {
        "AUC":
            roc_auc_score(
                y,
                scores,
            ),

        "KS":
            float(
                np.max(
                    tpr - fpr
                )
            ),

        "Recall":
            recall_score(
                y,
                prediction,
                zero_division=0,
            ),

        "Precision":
            precision_score(
                y,
                prediction,
                zero_division=0,
            ),

        "F1":
            f1_score(
                y,
                prediction,
                zero_division=0,
            ),

        "Accuracy":
            accuracy_score(
                y,
                prediction,
            ),

        "Brier":
            brier,

        "BSS":
            bss,

        "TP":
            int(tp),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "FN":
            int(fn),
    }


# ======================================================================
# 14. RANKING METRICS
# ======================================================================

def ranking_metrics(
    y,
    scores,
):

    frame = pd.DataFrame(
        {
            "y":
                np.asarray(
                    y,
                    dtype=int,
                ),

            "score":
                np.asarray(
                    scores,
                    dtype=float,
                ),
        }
    )


    frame = (
        frame
        .sort_values(
            "score",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    prevalence = float(
        frame[
            "y"
        ].mean()
    )


    positives = int(
        frame[
            "y"
        ].sum()
    )


    rows = []


    for pct in (
        10,
        20,
        30,
    ):

        n_contact = max(
            1,
            int(
                np.ceil(
                    len(frame)
                    * pct
                    / 100
                )
            ),
        )


        hits = int(
            frame[
                "y"
            ]
            .iloc[
                :n_contact
            ]
            .sum()
        )


        precision = (
            hits
            /
            n_contact
        )


        rows.append(
            {
                "top_pct":
                    pct,

                "n_contact":
                    n_contact,

                "hits":
                    hits,

                "precision":
                    precision,

                "lift":
                    precision
                    /
                    prevalence,

                "capture":
                    hits
                    /
                    positives,
            }
        )


    return rows


# ======================================================================
# 15. FIT ONE LEARNER / BRANCH
# ======================================================================

def fit_predict_branch(
    X,
    y,
    train_idx,
    test_idx,
    learner,
    seed,
    numeric_cols,
    categorical_cols,
):

    use_scaler = (
        learner == "LR"
    )


    X_train, X_test = (
        preprocess_branch(

            X,

            train_idx,

            test_idx,

            numeric_cols,

            categorical_cols,

            scale=
                use_scaler,
        )
    )


    model = make_model(
        learner,
        seed,
    )


    model.fit(
        X_train,
        y[
            train_idx
        ],
    )


    scores = (

        model.predict_proba(
            X_test
        )[:,1]

        .astype(float)
    )


    result = binary_metrics(
        y[
            test_idx
        ],
        scores,
    )


    del (
        model,
        X_train,
        X_test,
    )

    gc.collect()


    return (
        result,
        scores,
    )


# ======================================================================
# 16. PAIRED BOOTSTRAP
# ======================================================================

def paired_bootstrap_contrasts(
    y,
    predictions,
    seed,
):

    """
    Same resampled customer indices for all branches.

    Contrasts:
        B - A
        Bsym - A
        B - Bsym
    """


    y = np.asarray(
        y,
        dtype=int,
    )


    rng = np.random.default_rng(
        seed
    )


    pairs = [
        ("B", "A"),
        ("Bsym", "A"),
        ("B", "Bsym"),
    ]


    draws = {
        f"{left}-{right}": []
        for left, right
        in pairs
    }


    n = len(y)


    for i in range(
        N_BOOTSTRAP
    ):

        idx = rng.integers(
            0,
            n,
            size=n,
        )


        yb = y[
            idx
        ]


        if (
            yb.min()
            ==
            yb.max()
        ):
            continue


        aucs = {

            branch:
                roc_auc_score(
                    yb,
                    score[
                        idx
                    ],
                )

            for branch,
            score
            in predictions.items()
        }


        for left, right in pairs:

            draws[
                f"{left}-{right}"
            ].append(

                aucs[
                    left
                ]
                -
                aucs[
                    right
                ]
            )


    rows = []


    for left, right in pairs:

        name = (
            f"{left}-{right}"
        )


        values = np.asarray(
            draws[
                name
            ],
            dtype=float,
        )


        observed = (

            roc_auc_score(
                y,
                predictions[
                    left
                ],
            )

            -

            roc_auc_score(
                y,
                predictions[
                    right
                ],
            )
        )


        rows.append(
            {
                "contrast":
                    name,

                "auc_diff":
                    observed,

                "ci_lower":
                    float(
                        np.quantile(
                            values,
                            0.025,
                        )
                    ),

                "ci_upper":
                    float(
                        np.quantile(
                            values,
                            0.975,
                        )
                    ),

                "bootstrap_valid":
                    int(
                        len(
                            values
                        )
                    ),

                "n_diff_gt_0":
                    int(
                        np.sum(
                            values > 0
                        )
                    ),
            }
        )


    return (
        pd.DataFrame(
            rows
        ),

        pd.DataFrame(
            draws
        ),
    )


# ======================================================================
# 17. CANONICAL XGBOOST EVALUATION
# ======================================================================

def evaluate_canonical(
    branch_matrices,
    y,
    train_idx,
    test_idx,
    seed,
    label,
    out,
    bootstrap_seed,
):

    numeric_cols, categorical_cols = (
        infer_schema_from_A_train(

            branch_matrices[
                "A"
            ],

            train_idx,
        )
    )


    predictions = {}

    result_rows = []

    ranking_rows = []


    for branch in [
        "A",
        "Bsym",
        "B",
    ]:

        result, scores = (
            fit_predict_branch(

                branch_matrices[
                    branch
                ],

                y,

                train_idx,

                test_idx,

                learner=
                    "XGBoost",

                seed=
                    seed,

                numeric_cols=
                    numeric_cols,

                categorical_cols=
                    categorical_cols,
            )
        )


        predictions[
            branch
        ] = scores


        result_rows.append(
            {
                "Evaluation":
                    label,

                "Branch":
                    branch,

                "n_train":
                    len(
                        train_idx
                    ),

                "n_test":
                    len(
                        test_idx
                    ),

                "positive_train":
                    int(
                        y[
                            train_idx
                        ].sum()
                    ),

                "positive_test":
                    int(
                        y[
                            test_idx
                        ].sum()
                    ),

                **result,
            }
        )


        for row in ranking_metrics(
            y[
                test_idx
            ],
            scores,
        ):

            ranking_rows.append(
                {
                    "Evaluation":
                        label,

                    "Branch":
                        branch,

                    **row,
                }
            )


        print(
            f"{label} | {branch}: "
            f"AUC={result['AUC']:.6f} | "
            f"KS={result['KS']:.6f} | "
            f"Brier={result['Brier']:.6f}"
        )


    # ------------------------------------------------------------------
    # Paired bootstrap
    # ------------------------------------------------------------------

    contrast_df, draws_df = (
        paired_bootstrap_contrasts(

            y[
                test_idx
            ],

            predictions,

            seed=
                bootstrap_seed,
        )
    )


    contrast_df[
        "Evaluation"
    ] = label


    # ------------------------------------------------------------------
    # Kendall tau
    # ------------------------------------------------------------------

    tau_AB = kendalltau(

        predictions[
            "A"
        ],

        predictions[
            "B"
        ],

        nan_policy="omit",
    ).statistic


    tau_ABsym = kendalltau(

        predictions[
            "A"
        ],

        predictions[
            "Bsym"
        ],

        nan_policy="omit",
    ).statistic


    tau_BsymB = kendalltau(

        predictions[
            "Bsym"
        ],

        predictions[
            "B"
        ],

        nan_policy="omit",
    ).statistic


    tau_df = pd.DataFrame(
        [
            {
                "Evaluation":
                    label,

                "tau_A_B":
                    tau_AB,

                "tau_A_Bsym":
                    tau_ABsym,

                "tau_Bsym_B":
                    tau_BsymB,
            }
        ]
    )


    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------

    pd.DataFrame(
        result_rows
    ).to_csv(

        out
        / f"{label}_holdout_metrics.csv",

        index=False,
    )


    pd.DataFrame(
        ranking_rows
    ).to_csv(

        out
        / f"{label}_ranking.csv",

        index=False,
    )


    contrast_df.to_csv(

        out
        / f"{label}_paired_auc_contrasts.csv",

        index=False,
    )


    draws_df.to_csv(

        out
        / f"{label}_bootstrap_draws.csv",

        index=False,
    )


    tau_df.to_csv(

        out
        / f"{label}_kendall_tau.csv",

        index=False,
    )


    return (
        pd.DataFrame(
            result_rows
        ),
        contrast_df,
        tau_df,
        predictions,
    )


# ======================================================================
# 18. R50: FOUR LEARNERS × A/B/B-SYM
# ======================================================================

def run_R50(
    con,
    audit,
    A,
    B,
    y,
    admin_end,
    canonical_bsym,
    out,
):

    learners = [
        "LR",
        "RF",
        "LightGBM",
        "XGBoost",
    ]


    all_indices = np.arange(
        len(
            audit
        )
    )


    rows = []


    checkpoint_file = (
        out
        / "R50_four_learners_runs.csv"
    )


    for replicate_number, seed in enumerate(
        R50_SEEDS,
        start=1,
    ):

        print(
            "\n"
            + "=" * 100
        )

        print(
            f"R50 REPLICATE "
            f"{replicate_number}/50 | "
            f"split seed={seed}"
        )

        print(
            "=" * 100
        )


        # ----------------------------------------------------------
        # Same split for all learners / branches
        # ----------------------------------------------------------

        train_idx, test_idx = (
            train_test_split(

                all_indices,

                test_size=0.20,

                stratify=y,

                random_state=
                    seed,
            )
        )


        # ----------------------------------------------------------
        # B-sym must use positive TRAIN distribution for this seed.
        # Reuse canonical seed-42 B-sym to avoid duplicate query.
        # ----------------------------------------------------------

        if seed == CANONICAL_SPLIT_SEED:

            Bsym = (
                canonical_bsym.copy()
            )

        else:

            Bsym, _, bsym_info = (
                build_bsym_matrix(

                    con,

                    audit,

                    train_idx,

                    split_seed=
                        seed,

                    admin_end=
                        admin_end,
                )
            )


        branches = {
            "A":
                A,

            "Bsym":
                Bsym,

            "B":
                B,
        }


        # ----------------------------------------------------------
        # Shared schema inferred from A TRAIN only.
        # ----------------------------------------------------------

        numeric_cols, categorical_cols = (
            infer_schema_from_A_train(

                A,

                train_idx,
            )
        )


        for learner in learners:

            predictions = {}

            aucs = {}


            for branch in [
                "A",
                "Bsym",
                "B",
            ]:

                result, scores = (
                    fit_predict_branch(

                        branches[
                            branch
                        ],

                        y,

                        train_idx,

                        test_idx,

                        learner=
                            learner,

                        seed=
                            seed,

                        numeric_cols=
                            numeric_cols,

                        categorical_cols=
                            categorical_cols,
                    )
                )


                aucs[
                    branch
                ] = result[
                    "AUC"
                ]


                predictions[
                    branch
                ] = scores


            # ------------------------------------------------------
            # Window decomposition
            # ------------------------------------------------------

            d_window = (
                aucs["B"]
                -
                aucs["A"]
            )


            d_sym = (
                aucs["Bsym"]
                -
                aucs["A"]
            )


            d_asym = (
                aucs["B"]
                -
                aucs["Bsym"]
            )


            # numerical identity audit
            identity_error = (
                d_window
                -
                (
                    d_sym
                    +
                    d_asym
                )
            )


            # ------------------------------------------------------
            # Kendall tau
            # ------------------------------------------------------

            tau_A_B = kendalltau(

                predictions["A"],

                predictions["B"],

                nan_policy="omit",
            ).statistic


            tau_A_Bsym = kendalltau(

                predictions["A"],

                predictions["Bsym"],

                nan_policy="omit",
            ).statistic


            tau_Bsym_B = kendalltau(

                predictions["Bsym"],

                predictions["B"],

                nan_policy="omit",
            ).statistic


            row = {
                "replicate":
                    replicate_number,

                "seed":
                    seed,

                "learner":
                    learner,

                "n_train":
                    len(
                        train_idx
                    ),

                "n_test":
                    len(
                        test_idx
                    ),

                "positive_train":
                    int(
                        y[
                            train_idx
                        ].sum()
                    ),

                "positive_test":
                    int(
                        y[
                            test_idx
                        ].sum()
                    ),

                "AUC_A":
                    aucs[
                        "A"
                    ],

                "AUC_Bsym":
                    aucs[
                        "Bsym"
                    ],

                "AUC_B":
                    aucs[
                        "B"
                    ],

                "D_window_B_minus_A":
                    d_window,

                "D_sym_Bsym_minus_A":
                    d_sym,

                "D_asym_B_minus_Bsym":
                    d_asym,

                "decomposition_error":
                    identity_error,

                "Kendall_tau_A_B":
                    tau_A_B,

                "Kendall_tau_A_Bsym":
                    tau_A_Bsym,

                "Kendall_tau_Bsym_B":
                    tau_Bsym_B,
            }


            rows.append(
                row
            )


            print(
                f"{learner:9s} | "
                f"A={aucs['A']:.6f} | "
                f"Bsym={aucs['Bsym']:.6f} | "
                f"B={aucs['B']:.6f} | "
                f"Dwindow={d_window:+.6f} | "
                f"Dsym={d_sym:+.6f} | "
                f"Dasym={d_asym:+.6f} | "
                f"tau(A,B)={tau_A_B:.4f}"
            )


        # ----------------------------------------------------------
        # Save checkpoint after EACH seed.
        # ----------------------------------------------------------

        pd.DataFrame(
            rows
        ).to_csv(

            checkpoint_file,

            index=False,
        )


        del Bsym
        gc.collect()


    result_df = pd.DataFrame(
        rows
    )


    return result_df


# ======================================================================
# 19. R50 SUMMARY
# ======================================================================

def summarize_R50(
    result_df,
    out,
):

    metrics_to_summarize = [
        "AUC_A",
        "AUC_Bsym",
        "AUC_B",
        "D_window_B_minus_A",
        "D_sym_Bsym_minus_A",
        "D_asym_B_minus_Bsym",
        "Kendall_tau_A_B",
        "Kendall_tau_A_Bsym",
        "Kendall_tau_Bsym_B",
    ]


    rows = []


    for learner, group in (
        result_df.groupby(
            "learner"
        )
    ):

        for metric in metrics_to_summarize:

            values = (
                group[
                    metric
                ]
                .astype(float)
                .to_numpy()
            )


            row = {
                "learner":
                    learner,

                "metric":
                    metric,

                "R":
                    len(
                        values
                    ),

                "mean":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "sd":
                    float(
                        np.std(
                            values,
                            ddof=1,
                        )
                    ),

                "median":
                    float(
                        np.median(
                            values
                        )
                    ),

                "q025":
                    float(
                        np.quantile(
                            values,
                            0.025,
                        )
                    ),

                "q975":
                    float(
                        np.quantile(
                            values,
                            0.975,
                        )
                    ),
            }


            if metric.startswith(
                "D_"
            ):

                row[
                    "n_gt_0"
                ] = int(
                    np.sum(
                        values > 0
                    )
                )

            else:

                row[
                    "n_gt_0"
                ] = np.nan


            rows.append(
                row
            )


    summary = pd.DataFrame(
        rows
    )


    summary.to_csv(

        out
        / "R50_four_learners_summary.csv",

        index=False,
    )


    # --------------------------------------------------------------
    # Compact D-window table
    # --------------------------------------------------------------

    d_table = (

        summary[
            summary[
                "metric"
            ].isin(
                [
                    "D_window_B_minus_A",
                    "D_sym_Bsym_minus_A",
                    "D_asym_B_minus_Bsym",
                ]
            )
        ]

        .pivot(
            index="learner",
            columns="metric",
            values=[
                "mean",
                "sd",
                "q025",
                "q975",
                "n_gt_0",
            ],
        )
    )


    d_table.to_csv(

        out
        / "R50_Dwindow_decomposition_summary.csv"
    )


    tau_table = (

        summary[
            summary[
                "metric"
            ].str.startswith(
                "Kendall"
            )
        ]
        .copy()
    )


    tau_table.to_csv(

        out
        / "R50_Kendall_summary.csv",

        index=False,
    )


    return summary


# ======================================================================
# 20. MAIN
# ======================================================================

def main():

    stamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%dT%H%M%S%fZ"
        )
    )


    out = (

        OUTPUT_PARENT
        /
        f"run_{stamp}"
    )


    out.mkdir(
        parents=True,
        exist_ok=False,
    )


    work = (

        WORK_PARENT
        /
        f"santander_bsym_work_{stamp}"
    )


    work.mkdir(
        parents=True,
        exist_ok=False,
    )


    db_path = (
        work
        / "santander_bsym.duckdb"
    )


    con = duckdb.connect(
        str(
            db_path
        )
    )


    con.execute(
        "SET memory_limit='5GB'"
    )

    con.execute(
        f"SET threads={N_JOBS}"
    )


    spill = (
        work
        / "spill"
    )

    spill.mkdir(
        exist_ok=True
    )


    con.execute(
        f"SET temp_directory="
        f"{sql_string(spill)}"
    )


    print(
        "=" * 100
    )

    print(
        "SANTANDER A / B / B-SYM"
    )

    print(
        "S0 + R50 FOUR LEARNERS "
        "+ S1"
    )

    print(
        "=" * 100
    )

    print(
        "Raw:",
        RAW_FILE,
    )

    print(
        "Output:",
        out,
    )


    try:

        # ==========================================================
        # 20.1 Build common source + A/B
        # ==========================================================

        (
            audit,
            b,
            admin_end,
            source_stats,
            target_audit,
        ) = build_base_tables(

            con,
            out,
        )


        y = (
            audit[
                "TARGET_60D"
            ]
            .to_numpy(
                dtype=np.int8
            )
        )


        A = (
            audit[
                FEATURES
            ]
            .copy()
        )


        B = (
            b[
                FEATURES
            ]
            .copy()
        )


        all_indices = np.arange(
            len(
                audit
            )
        )


        # ==========================================================
        # 20.2 CANONICAL seed-42 split
        # ==========================================================

        canonical_train, canonical_test = (
            train_test_split(

                all_indices,

                test_size=0.20,

                stratify=y,

                random_state=
                    CANONICAL_SPLIT_SEED,
            )
        )


        split_frame = pd.DataFrame(
            {
                "NCODPERS":
                    audit[
                        "NCODPERS"
                    ],

                "TARGET_60D":
                    y,

                "SPLIT":
                    "train",
            }
        )


        split_frame.loc[
            canonical_test,
            "SPLIT",
        ] = "test"


        split_frame.to_csv(

            out
            / "canonical_seed42_split.csv",

            index=False,
        )


        print("\n" + "=" * 100)
        print("CANONICAL S0 SPLIT")
        print("=" * 100)

        print(
            f"Train       : "
            f"{len(canonical_train):,}"
        )

        print(
            f"Train +     : "
            f"{int(y[canonical_train].sum()):,}"
        )

        print(
            f"Holdout     : "
            f"{len(canonical_test):,}"
        )

        print(
            f"Holdout +   : "
            f"{int(y[canonical_test].sum()):,}"
        )


        # ==========================================================
        # 20.3 CANONICAL B-SYM
        # ==========================================================

        (
            canonical_bsym,
            canonical_cutoffs,
            canonical_bsym_info,
        ) = build_bsym_matrix(

            con,

            audit,

            canonical_train,

            split_seed=
                CANONICAL_SPLIT_SEED,

            admin_end=
                admin_end,

            save_cutoff_path=
                out
                / "S0_Bsym_cutoff_audit.csv",
        )


        pd.DataFrame(
            [
                canonical_bsym_info
            ]
        ).to_csv(

            out
            / "S0_Bsym_construction_summary.csv",

            index=False,
        )


        S0_branches = {
            "A":
                A,

            "Bsym":
                canonical_bsym,

            "B":
                B,
        }


        # ==========================================================
        # 20.4 S0 CANONICAL XGB
        # ==========================================================

        print("\n" + "=" * 100)
        print("S0 CANONICAL XGBOOST")
        print("=" * 100)


        (
            S0_metrics,
            S0_contrasts,
            S0_tau,
            S0_predictions,
        ) = evaluate_canonical(

            S0_branches,

            y,

            canonical_train,

            canonical_test,

            seed=
                CANONICAL_SPLIT_SEED,

            label=
                "S0",

            out=
                out,

            bootstrap_seed=
                BOOTSTRAP_SEED_S0,
        )


        # ==========================================================
        # 20.5 S1 COHORT
        # ==========================================================

        S1_mask = (

            audit[
                "N_FOLLOWUP_SNAPSHOTS"
            ]
            .to_numpy()
            > 0
        )


        S1_indices = np.where(
            S1_mask
        )[0]


        S1_train = np.asarray(
            [
                i
                for i
                in canonical_train

                if S1_mask[
                    i
                ]
            ],
            dtype=int,
        )


        S1_test = np.asarray(
            [
                i
                for i
                in canonical_test

                if S1_mask[
                    i
                ]
            ],
            dtype=int,
        )


        # All positives must remain.
        if (
            y[
                S1_indices
            ].sum()
            !=
            y.sum()
        ):

            raise AssertionError(
                "S1 removed positive customer."
            )


        print("\n" + "=" * 100)
        print("S1 COHORT")
        print("=" * 100)

        print(
            f"S0 N        : "
            f"{len(y):,}"
        )

        print(
            f"S1 N        : "
            f"{len(S1_indices):,}"
        )

        print(
            f"Excluded    : "
            f"{len(y)-len(S1_indices):,}"
        )

        print(
            f"S1 positive : "
            f"{int(y[S1_indices].sum()):,}"
        )

        print(
            f"S1 train    : "
            f"{len(S1_train):,}"
        )

        print(
            f"S1 holdout  : "
            f"{len(S1_test):,}"
        )


        pd.DataFrame(
            [
                {
                    "Cohort":
                        "S0",

                    "N":
                        len(
                            y
                        ),

                    "Positive":
                        int(
                            y.sum()
                        ),

                    "Train_n":
                        len(
                            canonical_train
                        ),

                    "Train_positive":
                        int(
                            y[
                                canonical_train
                            ].sum()
                        ),

                    "Holdout_n":
                        len(
                            canonical_test
                        ),

                    "Holdout_positive":
                        int(
                            y[
                                canonical_test
                            ].sum()
                        ),
                },

                {
                    "Cohort":
                        "S1",

                    "N":
                        len(
                            S1_indices
                        ),

                    "Positive":
                        int(
                            y[
                                S1_indices
                            ].sum()
                        ),

                    "Train_n":
                        len(
                            S1_train
                        ),

                    "Train_positive":
                        int(
                            y[
                                S1_train
                            ].sum()
                        ),

                    "Holdout_n":
                        len(
                            S1_test
                        ),

                    "Holdout_positive":
                        int(
                            y[
                                S1_test
                            ].sum()
                        ),
                },
            ]
        ).to_csv(

            out
            / "S0_S1_counts.csv",

            index=False,
        )


        # ==========================================================
        # 20.6 S1 A/B/B-SYM XGB
        #
        # S1 retains all positive customers.
        # Therefore positive training duration distribution is the
        # same as canonical S0 seed-42 training distribution.
        # Use same canonical B-sym matrix, subset via indices.
        # ==========================================================

        print("\n" + "=" * 100)
        print("S1 CANONICAL XGBOOST")
        print("=" * 100)


        (
            S1_metrics,
            S1_contrasts,
            S1_tau,
            S1_predictions,
        ) = evaluate_canonical(

            S0_branches,

            y,

            S1_train,

            S1_test,

            seed=
                CANONICAL_SPLIT_SEED,

            label=
                "S1",

            out=
                out,

            bootstrap_seed=
                BOOTSTRAP_SEED_S1,
        )


        # ==========================================================
        # 20.7 R50 FOUR LEARNERS
        # ==========================================================

        print("\n\n" + "=" * 100)
        print("START R50 — 4 LEARNERS × 3 BRANCHES")
        print("=" * 100)

        print(
            "Models to fit:",
            50 * 4 * 3,
        )


        R50_results = run_R50(

            con,

            audit,

            A,

            B,

            y,

            admin_end,

            canonical_bsym,

            out,
        )


        # ==========================================================
        # 20.8 R50 SUMMARY
        # ==========================================================

        R50_summary = summarize_R50(

            R50_results,

            out,
        )


        # ==========================================================
        # 20.9 RUN METADATA
        # ==========================================================

        metadata = {

            "source":
                str(
                    RAW_FILE
                ),

            "source_sha256":
                sha_file(
                    RAW_FILE
                ),

            "run_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "branches": {

                "A":
                    (
                        "fixed relationship-start + 60-day "
                        "feature cutoff"
                    ),

                "B":
                    (
                        "positive actual first post-day60 card "
                        "event cutoff; negative first-year/"
                        "administrative endpoint"
                    ),

                "Bsym":
                    (
                        "positive actual event cutoff; negative "
                        "pseudo duration sampled with replacement "
                        "from positive TRAIN event-duration "
                        "distribution of the same split, capped "
                        "at individual T_end"
                    ),
            },

            "missing_normalization": {

                "stage":
                    "before feature aggregation",

                "tokens_to_missing":
                    [
                        "",
                        "whitespace",
                        "NA",
                        "N/A",
                        "NAN",
                        "NULL",
                        "NONE",
                    ],

                "same_for_A_B_Bsym":
                    True,
            },

            "canonical_split_seed":
                CANONICAL_SPLIT_SEED,

            "R50_seeds":
                R50_SEEDS,

            "R":
                50,

            "learners":
                [
                    "LR",
                    "RF",
                    "LightGBM",
                    "XGBoost",
                ],

            "LR":
                {
                    "StandardScaler":
                        True,

                    "class_weight":
                        "balanced",

                    "max_iter":
                        2000,

                    "solver":
                        "lbfgs",
                },

            "RF":
                {
                    "n_estimators":
                        200,
                },

            "LightGBM":
                {
                    "n_estimators":
                        500,

                    "learning_rate":
                        0.05,

                    "num_leaves":
                        31,

                    "max_depth":
                        6,

                    "subsample":
                        0.8,

                    "colsample_bytree":
                        0.8,
                },

            "XGBoost":
                {
                    "n_estimators":
                        300,

                    "max_depth":
                        6,

                    "learning_rate":
                        0.05,

                    "subsample":
                        0.8,

                    "colsample_bytree":
                        0.8,

                    "tree_method":
                        "hist",

                    "max_bin":
                        256,
                },

            "bootstrap":
                {
                    "n":
                        N_BOOTSTRAP,

                    "paired":
                        True,

                    "contrasts":
                        [
                            "B-A",
                            "Bsym-A",
                            "B-Bsym",
                        ],
                },

            "decomposition":
                (
                    "D_window=(B-A); "
                    "D_sym=(Bsym-A); "
                    "D_asym=(B-Bsym); "
                    "D_window=D_sym+D_asym"
                ),

            "S1_rule":
                "N_FOLLOWUP_SNAPSHOTS > 0",

            "S1_preserves_seed42_split_membership":
                True,

            "target_audit":
                {
                    k:
                        (
                            int(v)
                            if isinstance(
                                v,
                                np.integer,
                            )
                            else v
                        )

                    for k, v
                    in target_audit.items()
                },

            "canonical_bsym":
                canonical_bsym_info,

            "environment": {

                "python":
                    platform.python_version(),

                "numpy":
                    np.__version__,

                "pandas":
                    pd.__version__,

                "scipy":
                    scipy.__version__,

                "sklearn":
                    sklearn.__version__,

                "xgboost":
                    xgboost.__version__,

                "lightgbm":
                    lightgbm.__version__,

                "duckdb":
                    duckdb.__version__,
            },
        }


        (
            out
            / "run_metadata.json"
        ).write_text(

            json.dumps(
                metadata,
                indent=2,
                ensure_ascii=False,
                default=str,
            ),

            encoding="utf-8",
        )


        # ==========================================================
        # 20.10 FINAL PRINT
        # ==========================================================

        print("\n\n" + "=" * 110)
        print("S0 CANONICAL METRICS")
        print("=" * 110)

        print(
            S0_metrics[
                [
                    "Branch",
                    "AUC",
                    "KS",
                    "Brier",
                    "BSS",
                    "Recall",
                    "Precision",
                    "F1",
                    "Accuracy",
                ]
            ]
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("S0 PAIRED BOOTSTRAP")
        print("=" * 110)

        print(
            S0_contrasts
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("S1 CANONICAL METRICS")
        print("=" * 110)

        print(
            S1_metrics[
                [
                    "Branch",
                    "AUC",
                    "KS",
                    "Brier",
                    "BSS",
                    "Recall",
                    "Precision",
                    "F1",
                    "Accuracy",
                ]
            ]
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("S1 PAIRED BOOTSTRAP")
        print("=" * 110)

        print(
            S1_contrasts
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("R50 — D-WINDOW DECOMPOSITION")
        print("=" * 110)

        print(
            R50_summary[
                R50_summary[
                    "metric"
                ].isin(
                    [
                        "D_window_B_minus_A",
                        "D_sym_Bsym_minus_A",
                        "D_asym_B_minus_Bsym",
                    ]
                )
            ][
                [
                    "learner",
                    "metric",
                    "R",
                    "mean",
                    "sd",
                    "q025",
                    "q975",
                    "n_gt_0",
                ]
            ]
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("R50 — KENDALL TAU")
        print("=" * 110)

        print(
            R50_summary[
                R50_summary[
                    "metric"
                ].str.startswith(
                    "Kendall"
                )
            ][
                [
                    "learner",
                    "metric",
                    "R",
                    "mean",
                    "sd",
                    "q025",
                    "q975",
                ]
            ]
            .round(6)
            .to_string(
                index=False
            )
        )


        print("\n" + "=" * 110)
        print("DONE")
        print("=" * 110)

        print(
            "Saved to:",
            out,
        )


    finally:

        try:
            con.close()
        except Exception:
            pass

        shutil.rmtree(
            work,
            ignore_errors=True,
        )


# ======================================================================
# 21. RUN
# ======================================================================

if __name__ == "__main__":
    main()

SANTANDER A / B / B-SYM
S0 + R50 FOUR LEARNERS + S1
Raw: /content/drive/MyDrive/Santander_CreditCard.csv
Output: /content/drive/MyDrive/Santander_AB_Bsym_S0_R50_S1/run_20261012T102347498317Z


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


TARGET AUDIT
n_rows              13619575
n_missing_target           0
n_invalid_target           0

Customers with multiple fecha_alta: 17


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (96 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LR        | A=0.841486 | Bsym=0.857996 | B=0.878771 | Dwindow=+0.037285 | Dsym=+0.016510 | Dasym=+0.020775 | tau(A,B)=0.7239
RF        | A=0.702316 | Bsym=0.748126 | B=0.860539 | Dwindow=+0.158223 | Dsym=+0.045810 | Dasym=+0.112414 | tau(A,B)=0.3467
LightGBM  | A=0.855511 | Bsym=0.877273 | B=0.965224 | Dwindow=+0.109713 | Dsym=+0.021762 | Dasym=+0.087951 | tau(A,B)=0.6125
XGBoost   | A=0.866193 | Bsym=0.883943 | B=0.970170 | Dwindow=+0.103977 | Dsym=+0.017750 | Dasym=+0.086228 | tau(A,B)=0.6628

R50 REPLICATE 49/50 | split seed=90


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LR        | A=0.838053 | Bsym=0.857709 | B=0.879551 | Dwindow=+0.041498 | Dsym=+0.019656 | Dasym=+0.021842 | tau(A,B)=0.7325
RF        | A=0.707276 | Bsym=0.759816 | B=0.884392 | Dwindow=+0.177116 | Dsym=+0.052540 | Dasym=+0.124576 | tau(A,B)=0.3408
LightGBM  | A=0.864540 | Bsym=0.876172 | B=0.962506 | Dwindow=+0.097967 | Dsym=+0.011632 | Dasym=+0.086335 | tau(A,B)=0.6249
XGBoost   | A=0.869531 | Bsym=0.883334 | B=0.965023 | Dwindow=+0.095491 | Dsym=+0.013803 | Dasym=+0.081688 | tau(A,B)=0.6901

R50 REPLICATE 50/50 | split seed=91
LR        | A=0.858146 | Bsym=0.866957 | B=0.892532 | Dwindow=+0.034386 | Dsym=+0.008812 | Dasym=+0.025574 | tau(A,B)=0.7148
RF        | A=0.757006 | Bsym=0.788226 | B=0.888394 | Dwindow=+0.131388 | Dsym=+0.031220 | Dasym=+0.100168 | tau(A,B)=0.3434
LightGBM  | A=0.861942 | Bsym=0.879246 | B=0.964705 | Dwindow=+0.102763 | Dsym=+0.017304 | Dasym=+0.085459 | tau(A,B)=0.6279
XGBoost   | A=0.879334 | Bsym=0.893700 | B=0.968857 | Dwindow=+0.089523 | Dsym=+0.014366